# Chebyshev Policies and the Mountain Car Problem: Reinforcement Learning for Low-dimensional Control Tasks
## Deriving the Optimum

Experiments for ICML 2026 paper "Chebyshev Policies and the Mountain Car Problem: Reinforcement Learning for Low-dimensional Control Tasks".  
In this notebook we perform experiments concerning the analytical/optimum solution.  
  
Version 2.0  
Date: 2026-01-29  
Current version: hannes.unger@fh-salzburg.ac.at, stefan.huber@fh-salzburg.ac.at     

# Description

Copied from [https://gymnasium.farama.org/environments/classic_control/mountain_car_continuous/](https://gymnasium.farama.org/environments/classic_control/mountain_car_continuous/):

<img width="1012" alt="image" src="https://gymnasium.farama.org/_images/mountain_car_continuous.gif">

This environment is part of the Classic Control environments which contains general information about the environment.

- Action Space: Box(-1.0, 1.0, (1,), float32)
- Observation Space / State Space: Box([-1.2 -0.07], [0.6 0.07], (2,), float32)

The Mountain Car MDP is a deterministic MDP that consists of a car placed stochastically at the bottom of a sinusoidal valley,  
with the only possible actions being the accelerations that can be applied to the car in either direction.  
The goal of the MDP is to strategically accelerate the car to reach the goal state on top of the right hill.  
There are two versions of the mountain car domain in gymnasium: one with discrete actions and one with continuous.  
This version is the one with continuous actions.  

## Observation Space

The observation is a ndarray with shape (2,) where the elements correspond to the following:  

| Num | Observation                          | Min  | Max |
| --- |--------------------------------------|------|-----|
| 0   | position of the car along the x-axis | -Inf | Inf |
| 1   | velocity of the car                  | -Inf | Inf |

## Action Space

The action is a ndarray with shape (1,), representing the directional force applied on the car.  
The action is clipped in the range [-1,1] and multiplied by a power of 0.0015.  

## Transition Dynamics:

Given an action, the mountain car follows the following transition dynamics:    
$velocity_{t+1} = velocity_{t} + force * self.power - 0.0025 * cos(3 * position_t)$  
  
$position_{t+1} = position_t + velocity_{t+1}$  
  
where force is the action clipped to the range [-1,1] and power is a constant 0.0015.  
The collisions at either end are inelastic with the velocity set to 0 upon collision with the wall.  
The position is clipped to the range [-1.2, 0.6] and velocity is clipped to the range [-0.07, 0.07].  

## Rewards

A negative reward of $-0.1 * action^2$ is received at each timestep to penalise for taking actions of large magnitude.  
If the mountain car reaches the goal then a positive reward of +100 is added to the negative reward for that timestep.  

## Starting State

The position of the car is assigned a uniform random value in [-0.6 , -0.4].  
The starting velocity of the car is always assigned to 0.   

## Episode End

The episode ends if either of the following happens:  
- Termination: The position of the car is greater than or equal to 0.45 (the goal position on top of the right hill)  
- Truncation: The length of the episode is 999.  

## Interpretation of Goal

Sparsity/delay of rewards is a major challenge for the RL agent.  
If exploration finds the goal flag early, the RL algorithm can converge to the global optimum.  
If it does not, the next best thing is convergence towards the local optimum of not applying any action.  

## Imports and Definitions

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec
import multiprocessing as mp
import matplotlib.gridspec as gridspec
from itertools import repeat
from gymnasium.wrappers import RecordVideo
from utils import parallel
from pickleshare import PickleShareDB
from algorithms import polynomial_agents
from utils.preprocessing import normalize, denormalize
from utils import plot

db = PickleShareDB('./picklesharedb')
EPSILON = 1e-3
UG_MIN_EPSILON = 1e-2
V_EPSILON = 1e-6

%load_ext autoreload
%autoreload 2

In [ ]:
def run_analytic_policy(c1 = 4.3345, c2=4.836, start=-0.6, steps=1000, left_wall=-1.2, minimum=-np.pi/6, render=False, record_video=False, single_phase=False):
    maxc = 1.0 / 0.07
    if c1 > maxc or c2 > maxc:
         raise Exception(f'Maximum C ({maxc}) exceeded')

    if record_video:
        base_env = gym.make("MountainCarContinuous-v0", render_mode="rgb_array")
        env = RecordVideo(base_env, video_folder="C:\\Users\\Hannes Waclawek\\Documents\\", episode_trigger = lambda x: True, disable_logger=True)
    elif render:
        env = gym.make("MountainCarContinuous-v0", render_mode="human")
    else:
        env = gym.make("MountainCarContinuous-v0")

    obs = [0.0, 0.0]
    actions = []
    velocities = []
    observations = []
    positions = []
    frames = []
    observations.append(env.reset(options={'low': start, 'high': start})[0])
    endgame = False

    r = 0.0
    c = c1

    for i in range(steps):
        if not endgame and np.fabs(observations[i][0] - left_wall) <= EPSILON:
            endgame = True
            if not single_phase:
                c=c2
        vel = observations[i][1]
        # if np.fabs(vel) < EPSILON and np.fabs(observations[i][0] - minimum) < EPSILON:
        #     action = 0.1
        # else:
        action = c * np.abs(vel)
        if np.fabs(observations[i][0] - minimum) < UG_MIN_EPSILON:
            action = max(0.1, action)
        if vel <= -0.0:
            action = -action
        actions.append(action)
        
        if render:
            env.render()
        if record_video:
            frames.append(env.render())
        obs, reward, terminated, truncated, info = env.step([action])
        observations.append(obs)
        velocities.append(obs[1])
        positions.append(obs[0])
        r += reward
        
        if terminated or truncated:
            break

    return r, obs[1], i+1, actions, observations


def get_analytic_action(obs, c, minimum=-np.pi/6):
    maxc = 1.0 / 0.07
    if c > maxc:
        raise Exception(f'Maximum C ({maxc}) exceeded')
    vel = obs[1]
    action = c * np.abs(vel)
    if np.fabs(obs[0] - minimum) < UG_MIN_EPSILON:
        action = max(0.1, action)
    if vel <= -0.0:
        action = -action
    
    return action


def plot_analytic_solution_heatmap(ax=None, fig=None, title=None, c1 = 0.405379, c2=0.9591, upper=[0.45, 0.07], lower=[-1.2, -0.07], trajectory=None, trajectory_label=None, xlim=(-1.2, 0.45), actionbar=True, fontsize=14, fontsize_title=18, xticks=None, yticks=None, yticksright=True):
    x = np.linspace(lower[0], upper[0], 100)
    y = np.linspace(lower[1], upper[1], 100)

    X, Y = np.meshgrid(x, y)
    states = np.vstack([X.ravel(), Y.ravel()]).T

    # Approximate "C2 border trajectory" with a polynomial
    c2_trajectory = parallel.job_analytic_solution_policy([0.0, c2], kwargs={'start_loc': lower[0]})[-1] # generate trajectory
    x, y = zip(*c2_trajectory) # separate the data into x and y lists
    coefficients = np.polyfit(x, y, 7) # Fit a 7th degree polynomial
    c2_border_poly = np.poly1d(coefficients)

    # Predict actions for each state
    actions = []
    for state in states:
        if state[1] < c2_border_poly(state[0]):
            c = c1
        else:
            c = c2
        actions.append(get_analytic_action(state, c))
    actions = np.array(actions)

    # Reshape to match the grid for heatmap plotting
    Z = actions.reshape(X.shape)

    levels=np.linspace(-1.0, 1.0, 21)

    # Plot the heatmap
    contourf = ax.contourf(X, Y, Z, levels=levels, cmap="viridis")  # Use a colormap like 'viridis' or 'plasma'

    if fig and actionbar:
        fig.colorbar(contourf, label="Action Magnitude")

    contour = ax.contour(X, Y, Z, levels=levels, cmap="viridis")  # Adjust levels for granularity
    #ax.clabel(contour, inline=True, fontsize=8)  # Add labels to the contour lines

    # Add a specific contour for the 0.0 level with custom styling
    zero_contour = ax.contour(X, Y, Z, levels=[0.0], colors="red", linewidths=2.5, zorder=5)
    
    if trajectory:
        ax.plot([row[0] for row in trajectory], [row[1] for row in trajectory], label=trajectory_label, color='white', zorder=4)
        ax.legend(loc='best', fontsize=fontsize)

    ax.set_xlabel(r"$x$", fontsize=fontsize+5)
    ax.set_ylabel(r"$\dot{x}$", fontsize=fontsize+5)
    ax.set_title(title, fontsize=fontsize_title)
    ax.set_xlim(xlim)
    ax.tick_params(axis='both', labelsize=fontsize)
    if xticks:
        ax.set_xticks(xticks)
    if yticks:
        ax.set_yticks(yticks)
    if yticksright:
        ax.yaxis.tick_right()
        #ax.yaxis.set_label_position("right")


def find_c1(x0=-0.6, c2=4.8358):
    v_max = 0.07
    maxc = 1.0 / v_max

    #Find C1, coarse search
    kwargs = {'start_loc': x0, 'stop_at_left_wall': False}

    c1 = np.linspace(1.0, maxc, 100)
    cins = [[c, c2] for c in c1]

    pool = mp.Pool(mp.cpu_count())
    allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

    cs = [a[0][0] for a in allres_c]
    r = [a[1] for a in allres_c]
    
    c1_index = np.nanargmax(r, axis=0)
    c1_coarse = cs[c1_index]
    lower_index = max(0, c1_index-1)
    upper_index = min(len(cs), c1_index+1)

    #Find C1, fine grained search
    kwargs = {'start_loc': x0, 'stop_at_left_wall': False}

    c1 = np.linspace(cs[lower_index], cs[upper_index], 1001)

    cins = [[c, c2] for c in c1]

    pool = mp.Pool(mp.cpu_count())
    allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

    cs = [a[0][0] for a in allres_c]
    r = [a[1] for a in allres_c]
    
    return cs[np.nanargmax(r, axis=0)]
    #return allres_c


def find_c2(left_wall=-1.2):
    v_max = 0.07
    maxc = 1.0 / v_max

    # Find C2, coarse search
    kwargs = {'start_loc': left_wall}

    c2 = np.linspace(1.0, maxc, 100)
    cins = [[0.0, c] for c in c2]

    pool = mp.Pool(mp.cpu_count())
    allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

    cs = [a[0][1] for a in allres_c]
    ls = [(100-a[1])/0.1 for a in allres_c]

    nan_indices = [i for i, x in enumerate(ls) if np.isnan(x)]
    if nan_indices:
        start_index = nan_indices[-1] + 1
    else:
        start_index = 0

    c2_coarse = cs[start_index]

    # Find C2, fine grained search
    c2 = np.linspace(c2_coarse-0.5, c2_coarse+0.5, 1001)
    cins = [[0.0, c] for c in c2]

    pool = mp.Pool(mp.cpu_count())
    allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

    cs = [a[0][1] for a in allres_c]
    ls = [a[1] for a in allres_c]

    nan_indices = [i for i, x in enumerate(ls) if np.isnan(x)]
    if nan_indices:
        start_index = nan_indices[-1] + 1
    else:
        start_index = 0

    return cs[start_index]

def find_single_phase_c(x0=-0.6):
    v_max = 0.07
    maxc = 1.0 / v_max

    #Find C, coarse search
    kwargs = {'start_loc': x0, 'stop_at_left_wall': False, 'single_phase': True}

    c1 = np.linspace(1.0, maxc, 100)
    cins = [[c, 0.0] for c in c1]

    pool = mp.Pool(mp.cpu_count())
    allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

    cs = [a[0][0] for a in allres_c]
    r = [a[1] for a in allres_c]
    
    c1_index = np.nanargmax(r, axis=0)
    lower_index = max(0, c1_index-1)
    upper_index = min(len(cs), c1_index+1)

    #Find C1, fine grained search
    kwargs = {'start_loc': x0, 'stop_at_left_wall': False, 'single_phase': True}

    c1 = np.linspace(cs[lower_index], cs[upper_index], 1001)

    cins = [[c, 0.0] for c in c1]

    pool = mp.Pool(mp.cpu_count())
    allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

    cs = [a[0][0] for a in allres_c]
    r = [a[1] for a in allres_c]
    
    return cs[np.nanargmax(r, axis=0)]


In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv
import os
import yaml
from huggingface_sb3 import load_from_hub
from stable_baselines3 import PPO
from rl_zoo3 import ALGOS, create_test_env, get_saved_hyperparams
from rl_zoo3.load_from_hub import download_from_hub
from rl_zoo3.utils import StoreDict, get_model_path
from stable_baselines3.common.utils import set_random_seed

# Retrieve the model from the hub
## repo_id = id of the model repository from the Hugging Face Hub (repo_id = {organization}/{repo_name})
## filename = name of the model zip file from the repository including the extension .zip
# checkpoint = load_from_hub(
#     repo_id="sb3/ppo-MountainCarContinuous-v0",
#     filename="ppo-MountainCarContinuous-v0.zip",
# )

def get_rl_zoo3_model_and_generate_env(algo="ppo", folder="rl-trained-agents", render=False):
    env_name="MountainCarContinuous-v0"
    organization="sb3"
    folder=folder
    exp_id=0
    repo_name=None
    force=False
    requires_download = False

    try:
        print("Loading model from local disk")
        _, model_path, log_path = get_model_path(
            exp_id,
            folder,
            algo,
            env_name
        )

        stats_path = os.path.join(log_path, env_name)
        hyperparams, maybe_stats_path = get_saved_hyperparams(stats_path, norm_reward=False, test_mode=True)
    except:
        print("Model not found on local disk, starting download")
        requires_download = True
    
    if requires_download:
        download_from_hub(
            algo=algo,
            env_name=env_name,
            organization=organization,
            folder=folder,
            exp_id=exp_id,
            repo_name=repo_name,
            force=force,
        )
        _, model_path, log_path = get_model_path(
            exp_id,
            folder,
            algo,
            env_name
        )

    
    stats_path = os.path.join(log_path, env_name)
    hyperparams, maybe_stats_path = get_saved_hyperparams(stats_path, norm_reward=False, test_mode=True)

    env_kwargs = {}
    args_path = os.path.join(log_path, env_name, "args.yml")
    if os.path.isfile(args_path):
        with open(args_path) as f:
            loaded_args = yaml.load(f, Loader=yaml.UnsafeLoader)
            if loaded_args["env_kwargs"] is not None:
                env_kwargs = loaded_args["env_kwargs"]

    env = create_test_env(
        env_name,
        n_envs=1,
        stats_path=maybe_stats_path,
        seed=set_random_seed(0),
        log_dir="logs/",
        should_render=render,
        hyperparams=hyperparams,
        env_kwargs=env_kwargs,
    )

    model = ALGOS[algo].load(model_path)

    return model, env


def run_rl_zoo3_model(model, env, start_loc=-np.pi/6, render=True):
    env.set_options({'low': start_loc, 'high': start_loc, 'render_mode': None})
    env.seed(seed=0)
    obs = env.reset()

    observations = []

    try:
        if env.norm_obs: # If observations are normalized, get "unnormalized" version
            print(f'start: {env.get_original_obs()}')
            observations.append(env.get_original_obs())
        else:
            print(f'start: {obs}')
            observations.append(obs)  
    except:
        print(f'start: {obs}')
        observations.append(obs)  

    try:
        if env.norm_reward:
            raise Exception("Normalized reward, aborting.")
    except:
        pass

    episode_reward = 0.0
    episode_rewards, episode_lengths = [], []
    ep_len = 0

    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, infos = env.step(action)

        if not done: # if done, obs will already be reset with SB3's VecEnv
            try:
                if env.norm_obs:
                    observations.append(env.get_original_obs())        
                else:
                    observations.append(obs)  
            except:
                observations.append(obs)  

        if render:
            env.render("human")

        episode_reward += reward[0]
        ep_len += 1

        if done:
            # NOTE: for env using VecNormalize, the mean reward
            # is a normalized reward when `--norm_reward` flag is passed
            print(f"Episode Reward: {episode_reward:.2f}")
            print("Episode Length", ep_len)
            episode_rewards.append(episode_reward)
            episode_lengths.append(ep_len)
            try:
                if env.norm_obs:
                    observations.append(np.array([infos[0]['terminal_observation']*np.sqrt(env.obs_rms.var + env.epsilon) + env.obs_rms.mean])) # denormalize value manually
                else:
                    observations.append(np.array([infos[0]['terminal_observation']]))
            except:
                observations.append(np.array([infos[0]['terminal_observation']]))
            # episode_reward = 0.0
            # ep_len = 0
            # env.set_options({'low': start_loc, 'high': start_loc})
            # env.seed(seed=0)
            # obs = env.reset()
            # print(f'start at {obs[0]}')

            return episode_reward, ep_len+1, observations

In [ ]:
def unnormalize_obs(obs, rms, epsilon):
    '''See _unnormalize_obs method in VecEnv'''
    obs * np.sqrt(env.obs_rms.var[0] + env.epsilon) + env.obs_rms.mean[0]

def plot_rl_zoo3_policy(env, model, ax=None, fig=None, title=None, unnormalized_ticks=False, trajectory=None, trajectory_label=None, save_to_file=None, actionbar=True, actionbaraxis=False, xlim=(-1.2, 0.45), fontsize=14, fontsize_title=18, xticks=None, yticks=None, hideyticks=False, yticksright=False, hideylabel=True):
    env.seed(seed=0)
    
    try:
        if env.norm_obs: 
            normalized = True
        else:
            normalized = False
    except:
        normalized = False

    if not normalized:
    # Discretize the state space
        x = np.linspace(env.observation_space.low[0], env.observation_space.high[0], 100)
        y = np.linspace(env.observation_space.low[1], env.observation_space.high[1], 100)
    else:
        # env.observation_space.low[0] and high[0] give original bounds, not normalized ones
        # Normalize bounds
        normalized_lower = env.normalize_obs([env.observation_space.low[0], env.observation_space.low[1]])
        normalized_upper = env.normalize_obs([env.observation_space.high[0], env.observation_space.high[1]])
        x = np.linspace(normalized_lower[0], normalized_upper[0], 100)
        y = np.linspace(normalized_lower[1], normalized_upper[1], 100)

    X, Y = np.meshgrid(x, y)
    states = np.vstack([X.ravel(), Y.ravel()]).T

    # Predict actions for each state
    actions = np.array([model.predict(state, deterministic=True)[0] for state in states])

    if unnormalized_ticks:
        x = np.linspace(env.observation_space.low[0], env.observation_space.high[0], 100)
        y = np.linspace(env.observation_space.low[1], env.observation_space.high[1], 100)
        X, Y = np.meshgrid(x, y)

    # Extract or compute the value to visualize (e.g., magnitude of action vector)
    # For a single action dimension, use `actions[:, 0]` or similar.
    #action_magnitude = np.linalg.norm(actions, axis=1)
    action_magnitude = actions[:, 0]

    # Reshape to match the grid for heatmap plotting
    Z = action_magnitude.reshape(X.shape)

    levels=np.linspace(-1.0, 1.0, 21)

    # Plot the heatmap
    contourf = ax.contourf(X, Y, Z, levels=levels, cmap="viridis")  # Use a colormap like 'viridis' or 'plasma'

    if fig and actionbar:
        if actionbaraxis:
            cbar = fig.colorbar(contourf, cax=actionbaraxis, label="Action Magnitude")
            cbar.ax.set_ylabel("Action Magnitude", fontsize=fontsize)
            cbar.ax.tick_params(labelsize=fontsize)        
        else:
            cbar = fig.colorbar(contourf, label="Action Magnitude")
            cbar.ax.set_ylabel("Action Magnitude", fontsize=fontsize)
            cbar.ax.tick_params(labelsize=fontsize)

    contour = ax.contour(X, Y, Z, levels=levels, cmap="viridis")  # Adjust levels for granularity
    #ax.clabel(contour, inline=True, fontsize=8)  # Add labels to the contour lines

    # Add a specific contour for the 0.0 level with custom styling
    zero_contour = ax.contour(X, Y, Z, levels=[0.0], colors="red", linewidths=2.5, zorder=5)

    if trajectory:
        ax.plot([row[0][0] for row in trajectory], [row[0][1] for row in trajectory], label=trajectory_label, color='white', zorder=4)
        ax.legend(loc='best', fontsize=fontsize)

    ax.set_xlabel(r"$x$", fontsize=fontsize+5)
    if not hideylabel:
        ax.set_ylabel(r"$\dot{x}$", fontsize=fontsize+5)
    ax.set_title(title, fontsize=fontsize_title)
    ax.set_xlim(xlim)
    ax.tick_params(axis='both', labelsize=fontsize)
    if xticks:
        ax.set_xticks(xticks)
    if yticks:
        ax.set_yticks(yticks)
    if hideyticks:
        ax.tick_params(axis='y', which='both', labelleft=False)
    else:
        if yticksright:
            ax.yaxis.tick_right()

    if save_to_file:
        plt.savefig(save_to_file)

    if fig and actionbar and not actionbaraxis:
        return cbar

In [ ]:
def read_matlab_action_export(file=None):
    # Open the file and read lines
    with open(file, 'r') as file:
        # Strip whitespaces
        float_array = [float(line.strip()) for line in file]

    return float_array

def run_fixed_action_sequence(actions, start=-0.6, steps=1000, render=False):
    if render:
        env = gym.make("MountainCarContinuous-v0", render_mode="human")
    else:
        env = gym.make("MountainCarContinuous-v0")

    obs = [0.0, 0.0]
    observations = []
    observations.append(env.reset(options={'low': start, 'high': start})[0])

    r = 0.0

    for i in range(steps):   
        if render:
            env.render()
        obs, reward, terminated, truncated, info = env.step([actions[i]])
        observations.append(obs)
        r += reward
        
        if terminated or truncated:
            break

    return r, obs[1], i+1, observations

In [ ]:
def denormalize_sb3_ch_trajectory(observations):
    denormalized_observations = []
    for o in observations:
        denormalized_observations.append([denormalize(o[0], max_value=np.array([0.6, 0.07]), min_value=np.array([-1.2, -0.07]))])
    return denormalized_observations

## Evaluate state-of-the-art agents

We take pre-trained rl-baselines3-zoo agents from [3] hosted at [4].  
Refer to benchmarks from [5].  
There is one line in gymnasium\envs\classic_control\continuous_mountain_car.py to consider specifically:    
  
```python
        self.goal_position = (
            0.45  # was 0.5 in gymnasium, 0.45 in Arnaud de Broissia's version
        )
```
  
The current master branch of Gymnasium at [1] at commit 75bd3be says: “was 0.5 in gym, 0.45 in Arnaud de Broissia’s version”.      
This comment goes back all the way to the predecessor library Gym commit c97551e "added continuous mountain car v0" at Aug 24, 2016 at [2],   
where the environment was adapted from the "FAReinforcement" library.  
continuous_mountain_car.py environment constraints have not changed between Gymnasium commit 75bd3be and gym commit c97551e.  
rl-baselines3-zoo (rl_zoo3\train.py) added Gymnasium support on Apr 14, 2023 with commit 73b1712 and utilized gym before this, with the first commit 5361ba6 dating back to Oct 14, 2019.   
This means that, although agents were potentially trained using the gym library instead of Gymnasium, constraints are still aligned as they have not changed between concerned versions.   
  
[1] https://github.com/Farama-Foundation/Gymnasium  
[2] https://github.com/openai/gym  
[3] https://github.com/DLR-RM/rl-baselines3-zoo  
[4] https://huggingface.co/sb3  
[5] https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/benchmark.md  

### Deterministic starting position $x_0=-0.6$

In [ ]:
start = -0.6
#start = -np.pi/6

In [ ]:
ppo_model, ppo_env = get_rl_zoo3_model_and_generate_env('ppo')

In [ ]:
ppo_episode_reward, ppo_ep_len, ppo_v_target = run_rl_zoo3_model(ppo_model, ppo_env, start_loc=start)

db['ppo_episode_reward'] = ppo_episode_reward
db['ppo_episode_len'] = ppo_ep_len
db['ppo_v_target'] = ppo_v_target[-1][0][1]

In [ ]:
tqc_model, tqc_env = get_rl_zoo3_model_and_generate_env('tqc')

In [ ]:
tqc_episode_reward, tqc_ep_len, tqc_v_target = run_rl_zoo3_model(tqc_model, tqc_env, start_loc=start)

db['tqc_episode_reward'] = tqc_episode_reward
db['tqc_episode_len'] = tqc_ep_len
db['tqc_v_target'] = tqc_v_target[-1][0][1]

In [ ]:
sac_model, sac_env = get_rl_zoo3_model_and_generate_env('sac')

In [ ]:
sac_episode_reward, sac_ep_len, sac_v_target = run_rl_zoo3_model(sac_model, sac_env, start_loc=start)

db['sac_episode_reward'] = sac_episode_reward
db['sac_episode_len'] = sac_ep_len
db['sac_v_target'] = sac_v_target[-1][0][1]

In [ ]:
td3_model, td3_env = get_rl_zoo3_model_and_generate_env('td3')

In [ ]:
td3_episode_reward, td3_ep_len, td3_v_target = run_rl_zoo3_model(td3_model, td3_env, start_loc=start)

db['td3_episode_reward'] = td3_episode_reward
db['td3_episode_len'] = td3_ep_len
db['td3_v_target'] = td3_v_target[-1][0][1]

In [ ]:
trpo_model, trpo_env = get_rl_zoo3_model_and_generate_env('trpo')

In [ ]:
trpo_episode_reward, trpo_ep_len, trpo_v_target = run_rl_zoo3_model(trpo_model, trpo_env, start_loc=start)

db['trpo_episode_reward'] = trpo_episode_reward
db['trpo_episode_len'] = trpo_ep_len
db['trpo_v_target'] = trpo_v_target[-1][0][1]

In [ ]:
a2c_model, a2c_env = get_rl_zoo3_model_and_generate_env('a2c')

In [ ]:
a2c_episode_reward, a2c_ep_len, a2c_v_target = run_rl_zoo3_model(a2c_model, a2c_env, start_loc=start)

db['a2c_episode_reward'] = a2c_episode_reward
db['a2c_episode_len'] = a2c_ep_len
db['a2c_v_target'] = a2c_v_target[-1][0][1]

In [ ]:
ddpg_model, ddpg_env = get_rl_zoo3_model_and_generate_env('ddpg')

In [ ]:
ddpg_episode_reward, ddpg_ep_len, ddpg_v_target = run_rl_zoo3_model(ddpg_model, ddpg_env, start_loc=start)

db['ddpg_episode_reward'] = ddpg_episode_reward
db['ddpg_episode_len'] = ddpg_ep_len
db['ddpg_v_target'] = ddpg_v_target[-1][0][1]

In [ ]:
ars_model, ars_env = get_rl_zoo3_model_and_generate_env('ars')

In [ ]:
ars_episode_reward, ars_ep_len, ars_v_target = run_rl_zoo3_model(ars_model, ars_env, start_loc=start)

db['ars_episode_reward'] = ars_episode_reward
db['ars_episode_len'] = ars_ep_len
db['ars_v_target'] = ars_v_target[-1][0][1]

### Analyze ars model

In [ ]:
ars_model, ars_env = get_rl_zoo3_model_and_generate_env('ars')

In [ ]:
plot_rl_zoo3_policy(ars_env, ars_model, "Policy Heatmap: Augmented Random Search (ARS)\n (Unnormalized)", unnormalized_ticks=True)

In [ ]:
reward, _, observations = run_rl_zoo3_model(ars_model, ars_env, start_loc=-0.6)

In [ ]:
#plot_rl_zoo3_policy(ars_env, ars_model, "Policy Heatmap: Augmented Random Search (ARS)\n (Unnormalized)", unnormalized_ticks=True, trajectory=observations, trajectory_label=f'Trajectory starting at x0 = {x0}\nReward: {reward:.4f}', save_to_file=".\\paper-figures\\policyheatmap_ars_trajectory_x0.pdf")
fig, ax = plt.subplots()
fig.set_figwidth(10)
fig.set_figheight(8)
plot_rl_zoo3_policy(ars_env, ars_model, ax=ax, fig=fig, unnormalized_ticks=True, trajectory=observations, trajectory_label=f'Trajectory starting at x0 = {start}\nReward: {reward:.4f}', save_to_file="policyheatmap_ars_trajectory_x0.pdf")

## Analytic Policy

Our unconstrained analytic solution describes the action $a(t)$ solely based on the current velocity $\dot{x}$ as  

$a(t)= C * a_{max}\cdot\text{sign}(\dot{x})\cdot\sqrt{|\dot{x}|}$,   

with  

$C_{max} = \frac{1}{\sqrt{v_{max}}}$.  
  
Generally, we divide the sequence into two distinct phases:
 - In phase 1, we apply $C_1$, building up potential until reaching the left wall  
 - In phase 2, we apply $C_2$ for reaching the goal flag with the least amount of remaining velocity possible

### Naive empirical approach of finding $C_1$, $C_2$

In a first attempt, we seek to find $C_1$ and $C_2$ empirically, by plotting loss $l$, excessive velocity $\dot{x}(t_*)$ and total distance over different $C$ s, with    
$l = 100 - R$  

##### Find $C_2$ by starting from left wall

In [ ]:
start_loc = -1.2
kwargs = {'start_loc': start_loc}
v_max = 0.07
maxc = 1.0 / v_max

c2 = np.linspace(4.5, 5, 1001)

cins = [[0.0, c] for c in c2]

In [ ]:
pool = mp.Pool(mp.cpu_count())
allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

cs = [a[0][1] for a in allres_c]
ls = [(100-a[1]) for a in allres_c]
total_distances = [a[2] for a in allres_c]
target_velocities = [a[3] for a in allres_c]
C_squ = np.array(ls)/np.array(total_distances)
steps = [len(a[4]) for a in allres_c if isinstance(a[4], list)]

nan_indices = [i for i, x in enumerate(ls) if np.isnan(x)]
if nan_indices:
    start_index = nan_indices[-1] + 1
else:
    start_index = 0

db["c2_optimum"] = cs[start_index]

In [ ]:
from matplotlib.lines import Line2D

c = db["c2_optimum"]

fig, ax1 = plt.subplots()
fig.set_figwidth(18)

ax1.plot(cs, ls, '.', label=r'$l$')
ax1.plot(cs, total_distances, '.', label=r'$\xi_*-\xi_0$')
ax1.set_xlabel(r'$C_2$')
ax1.set_ylabel(r'$l, (\xi_*-\xi_0)$')

ax2 = ax1.twinx()
ax2.plot(cs, target_velocities, '.r', label=r'$\dot{x}(t_*)$')
ax2.set_xlabel(r'$C_2$')
ax2.set_ylabel(r'$\dot{x}(t_*)$')

fig.legend(loc='lower center', bbox_to_anchor=(0.17, 0.68))
# fig.tight_layout()
fig.suptitle(r'Loss $l$, excessive velocity $\dot{x}(t_*)$ and total distance $\xi_*-\xi_0$ over $C_2$.' '\n' r'First $C_2$ reaching target is' f' {c:.4f} with target velocity {target_velocities[start_index]:.4f}.' '\n' f'Left wall at x=-1.2, starting at x_0={start_loc}.', y=1.1)  # otherwise the right y-label is slightly clipped

##### Find $C_1$ for $x_0=-0.6$ with found $C_2$

In [ ]:
start_loc = -0.6
kwargs = {'start_loc': start_loc}
v_max = 0.07
maxc = 1.0 / v_max

c1 = np.linspace(0.001, maxc, 1001)
c2 = db["c2_optimum"]

cins = [[c, c2] for c in c1]

In [ ]:
pool = mp.Pool(mp.cpu_count())
allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

cs = [a[0][0] for a in allres_c]
ls = [(100-a[1]) for a in allres_c]
total_distances = [a[2] for a in allres_c]
target_velocities = [a[3] for a in allres_c]
C_squ = np.array(ls)/np.array(total_distances)
steps = [len(a[4]) for a in allres_c if isinstance(a[4], list)]

nan_indices = [i for i, x in enumerate(ls) if np.isnan(x)]
if nan_indices:
    start_index = nan_indices[-1] + 1
else:
    start_index = 0

db["c1_optimum"] = cs[np.nanargmin(ls, axis=0)]
db["c1_optimum"] 

In [ ]:
c = db["c1_optimum"]

fig, ax1 = plt.subplots()
fig.set_figwidth(18)

ax1.plot(cs, ls, '.', label=r'$l$')
ax1.plot(cs, total_distances, '.', label=r'$\xi_*-\xi_0$')
ax1.set_xlabel(r'$C_1$')
ax1.set_ylabel(r'$l, (\xi_*-\xi_0)$')

ax2 = ax1.twinx()
ax2.plot(cs, target_velocities, '.r', label=r'$\dot{x}(t_*)$')
ax2.set_xlabel(r'$C_1$')
ax2.set_ylabel(r'$\dot{x}(t_*)$')

fig.legend(loc='lower center', bbox_to_anchor=(0.17, 0.68))
# fig.tight_layout()
fig.suptitle(r'Loss $l$, excessive velocity $\dot{x}(t_*)$ and total distance $\xi_*-\xi_0$ over $C_1$.' '\n' r'Minimum loss reached for $C_1$ =' f' {c:.4f} with target velocity {target_velocities[start_index]:.4f}.' '\n' r'Left wall at $x=-1.2$, starting at $x_0=$' f'{start_loc},' r' $C_2=$' f'{c2}', y=1.1)  # otherwise the right y-label is slightly clipped

### Define C1, C2 for $x_0=-0.6$

In [ ]:
c2 = find_c2()
db["c2_optimum"] = c2
c2

In [ ]:
c1 = find_c1(-0.6, db["c2_optimum"])
db["c1_optimum"] = c1
c1

### Plotting analytic policy for different $x_0$

In [ ]:
x0=-0.6
c1 = db["c1_optimum"]
c2 = db["c2_optimum"]
res = run_analytic_policy(c1=c1, c2=c2, start=x0)
observations = res[-1]
reward = res[0]

fig, ax1 = plt.subplots() 
fig.set_figwidth(10)
fig.set_figheight(8)
#plot_analytic_solution_heatmap(ax=ax1, fig=fig, title=f'Analytic Solution Policy Heatmap for x0 = {x0} (C1 = {c1:.4f}, C2={c2:.4f})', c1=c1, c2=c2, trajectory=observations, trajectory_label=f'Trajectory starting at x0 = {x0}\nReward: {reward:.4f}')
plot_analytic_solution_heatmap(ax=ax1, fig=fig, c1=c1, c2=c2, trajectory=observations, trajectory_label=f'Trajectory starting at x0 = {x0}\nReward: {reward:.4f}')
fig.savefig(".\\paper-figures\\policyheatmap_analytic_trajectory_x0.pdf")

In [ ]:
x0 = -np.pi/6
c1 = find_c1(x0)
res = run_analytic_policy(c1, c2, start=x0)
observations = res[-1]
reward = res[0]

fig, ax1 = plt.subplots() 
fig.set_figwidth(10)
fig.set_figheight(8)
plot_analytic_solution_heatmap(ax=ax1, fig=fig, title=f'Analytic Solution Policy Heatmap for x0 = {x0} (C1 = {c1:.4f}, C2={c2:.4f})', c1=c1, c2=c2, trajectory=observations, trajectory_label=f'Trajectory starting at x0 = {x0}\nReward: {reward:.4f}')

In [ ]:
x0 = -0.4
c1 = find_c1(x0, c2)
res = run_analytic_policy(c1, c2, start=x0)
observations = res[-1]
reward = res[0]

fig, ax1 = plt.subplots() 
fig.set_figwidth(10)
fig.set_figheight(8)
plot_analytic_solution_heatmap(ax=ax1, fig=fig, title=f'Analytic Solution Policy Heatmap for x0 = {x0} (C1 = {c1:.4f}, C2={c2:.4f})', c1=c1, c2=c2, trajectory=observations, trajectory_label=f'Trajectory starting at x0 = {x0}\nReward: {reward:.4f}')

## Determining $\pi_{\text{ana}}$ 

Worst-case C for $x_0\in[-0.6, 0.4]$.

In [ ]:
x0s = np.linspace(-0.6, -0.4, 100)

In [ ]:
c1s = []
for x in x0s:
    c1 = find_c1(x, db["c2_optimum"])
    c1s.append(c1)
db["c1s_linspace_100"] = c1s

In [ ]:
c1s = db["c1s_linspace_100"]

In [ ]:
x0s_diracs = np.linspace(-0.6, -0.4, 1000)

In [ ]:
ls_linear = db["analytic_loss_over_x0"]
c1s_linear = db["analytic_cs_over_x0"]

ls_sqrt = db["analytic_loss_over_x0_sqrt"]
c1s_sqrt = db["analytic_loss_over_x0_sqrt"]

ls_corrected = db["analytic_loss_over_x0_corrected"]
c1s_corrected = db["analytic_cs_over_x0_corrected"]

fig, ax1 = plt.subplots()
fig.set_figwidth(9)

ax1.plot(x0s_diracs, ls_linear, '.', label='l with $0.1$ diracs')
ax1.plot(x0s, ls_sqrt, '.', label='l with sqrt(abs(vel)))')
ax1.plot(x0s, ls_corrected, '.', label='l with bootstrapping only in beginning')
ax1.set_xlabel(r'$x_0$')
ax1.set_ylabel(r'$\ell$')
ax1.axvline(-np.pi/6, color ='k', linestyle='--', lw = 2, alpha = 0.75, label=r'$-\frac{\pi}{6}$')
ax1.legend(loc='best')

#fig.legend(loc='lower center', bbox_to_anchor=(0.17, 0.68))
fig.tight_layout()
fig.suptitle(r'Analytic solution: Loss $\ell$ over different $x_0$' , y=1.1)  # otherwise the right y-label is slightly clipped

In [ ]:
db["c1_pi_ana"] = np.max(c1s)
c1s = db["analytic_cs_over_x0"]
# np.max(c1s)
# db["c1_max"] = np.max(c1s)
np.max(c1s)

In [ ]:
db["c1_pi_ana"]

### Why are there outliers along the way?

In [ ]:
c1s[34]

In [ ]:
c1s[35]

In [ ]:
run_analytic_policy()

In [ ]:
c1=c1s[34]
c2=db['c2_optimum']
x0=x0s[34]

res = run_analytic_policy(c1=c1, c2=c2, start=x0, render=True)
observations = res[-1]
reward = res[0]

fig, ax1 = plt.subplots() 
fig.set_figwidth(10)
fig.set_figheight(8)
plot_analytic_solution_heatmap(ax=ax1, fig=fig, title=f'Analytic Solution Policy Heatmap for x0 = {x0:.4f} (C1 = {c1:.4f}, C2={c2:.4f})', c1=c1, c2=c2, trajectory=observations, trajectory_label=f'Trajectory starting at x0 = {x0}\nReward: {reward:.4f}')

In [ ]:
c1=c1s[35]
c2=db['c2_optimum']
x0=x0s[35]

res = run_analytic_policy(c1=c1, c2=c2, start=x0, render=True)
observations = res[-1]
reward = res[0]

fig, ax1 = plt.subplots() 
fig.set_figwidth(10)
fig.set_figheight(8)
plot_analytic_solution_heatmap(ax=ax1, fig=fig, title=f'Analytic Solution Policy Heatmap for x0 = {x0} (C1 = {c1:.4f}, C2={c2:.4f})', c1=c1, c2=c2, trajectory=observations, trajectory_label=f'Trajectory starting at x0 = {x0}\nReward: {reward:.4f}')

## Single-phase vs. Two-phase solutions

### If we move the left wall to $\hat{x*}$...
When is a single-phase policy with target velocity close to $0$ feasible?  
... $\exists x_0, \exists k: x_{k-1} < x_{min}?$  

*Attention*: Modification necessary: self.min_position = -2*np.pi/6-0.45 in continuous_mountain_car.py

In [ ]:
start_loc = -0.6
kwargs = {'start_loc': start_loc, 'single_phase': True}
v_max = 0.07
maxc = 1.0 / v_max

c = np.linspace(0.01, maxc, 1001)

cins = [[c, 0.0] for c in c]

pool = mp.Pool(mp.cpu_count())
allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

cs = [a[0][0] for a in allres_c]
ls = [a[1] for a in allres_c]
target_velocities = [a[3] for a in allres_c]
obs = [a[4] for a in allres_c]

nan_indices = [i for i, x in enumerate(ls) if np.isnan(x)]
if nan_indices:
    start_index = nan_indices[-1] + 1
else:
    start_index = 0

pos = [[[t[0] for t in a], target_velocities[i], cs[i]] for i, a in enumerate(obs) if isinstance(a, list)]
minpos = [[min(a[0]), a[1], a[2]] for a in pos]

In [ ]:
fig, ax1 = plt.subplots()
fig.set_figwidth(9)

ax1.plot([row[2] for row in minpos], [row[0] for row in minpos], '.r', label=r'$x_{k-1}$')
ax1.set_xlabel(r'$C$')
ax1.set_ylabel(r'$x_{k-1}$')
plt.axhline(y=-1.2, c='r', linestyle='dotted', label=r'$x_{min}$', alpha=0.5)
plt.axhline(y=-0.6, c='r', linestyle='dashdot', label=r'$x_{0}$', alpha=0.5)

ax2 = ax1.twinx()
ax2.plot([row[2] for row in minpos], [row[1] for row in minpos], '.', label=r'$\dot{x}(t_*)$')
ax2.set_xlabel(r'$C$')
ax2.set_ylabel(r'$\dot{x}(t_*)$')

fig.legend(loc='upper left', bbox_to_anchor=(0.14, 0.80))
# fig.tight_layout()
#fig.suptitle(r'$x_{k-1}$ and excessive velocity $\dot{x}(t_*)$ over $C$ at $x_0=$' f' {start_loc}.' '\n' r'Left wall at $x~=-1.5$', y=1.1)  # otherwise the right y-label is slightly clipped
plt.savefig(".\\paper-figures\\single_phase_c.pdf")

In [ ]:
start_loc = -np.pi/6
kwargs = {'start_loc': start_loc, 'single_phase': True}
v_max = 0.07
maxc = 1.0 / v_max

c = np.linspace(0.01, maxc, 1001)

cins = [[c, c] for c in c]

pool = mp.Pool(mp.cpu_count())
allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

cs = [a[0][0] for a in allres_c]
ls = [a[1] for a in allres_c]
target_velocities = [a[3] for a in allres_c]
obs = [a[4] for a in allres_c]

nan_indices = [i for i, x in enumerate(ls) if np.isnan(x)]
if nan_indices:
    start_index = nan_indices[-1] + 1
else:
    start_index = 0

pos = [[[t[0] for t in a], target_velocities[i], cs[i]] for i, a in enumerate(obs) if isinstance(a, list)]
minpos = [[min(a[0]), a[1], a[2]] for a in pos]

In [ ]:
fig, ax1 = plt.subplots()
fig.set_figwidth(9)

ax1.plot([row[2] for row in minpos], [row[0] for row in minpos], '.r', label=r'$x_{k-1}$')
ax1.set_xlabel(r'$C$')
ax1.set_ylabel(r'$x_{k-1}$')
plt.axhline(y=-1.2, c='r', linestyle='dotted', label=r'$x_{min}$', alpha=0.5)
plt.axhline(y=-0.6, c='r', linestyle='dashdot', label=r'$x_{0}$', alpha=0.5)

ax2 = ax1.twinx()
ax2.plot([row[2] for row in minpos], [row[1] for row in minpos], '.', label=r'$\dot{x}(t_*)$')
ax2.set_xlabel(r'$C$')
ax2.set_ylabel(r'$\dot{x}(t_*)$')

fig.legend(loc='upper left', bbox_to_anchor=(0.14, 0.80))
# fig.tight_layout()
fig.suptitle(r'$x_{k-1}$ and excessive velocity $\dot{x}(t_*)$ over $C$ at $x_0=$' f' {start_loc}.' '\n' r'Left wall at $x~=-1.5$', y=1.1)  # otherwise the right y-label is slightly clipped

In [ ]:
start_loc = -0.4
kwargs = {'start_loc': start_loc, 'single_phase': True}
v_max = 0.07
maxc = 1.0 / v_max

c = np.linspace(0.01, maxc, 1001)

cins = [[c, c] for c in c]

pool = mp.Pool(mp.cpu_count())
allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

cs = [a[0][0] for a in allres_c]
ls = [a[1] for a in allres_c]
target_velocities = [a[3] for a in allres_c]
obs = [a[4] for a in allres_c]

nan_indices = [i for i, x in enumerate(ls) if np.isnan(x)]
if nan_indices:
    start_index = nan_indices[-1] + 1
else:
    start_index = 0

pos = [[[t[0] for t in a], target_velocities[i], cs[i]] for i, a in enumerate(obs) if isinstance(a, list)]
minpos = [[min(a[0]), a[1], a[2]] for a in pos]

In [ ]:
fig, ax1 = plt.subplots()
fig.set_figwidth(9)

ax1.plot([row[2] for row in minpos], [row[0] for row in minpos], '.r', label=r'$x_{k-1}$')
ax1.set_xlabel(r'$C$')
ax1.set_ylabel(r'$x_{k-1}$')
plt.axhline(y=-1.2, c='r', linestyle='dotted', label=r'$x_{min}$', alpha=0.5)
plt.axhline(y=-0.6, c='r', linestyle='dashdot', label=r'$x_{0}$', alpha=0.5)

ax2 = ax1.twinx()
ax2.plot([row[2] for row in minpos], [row[1] for row in minpos], '.', label=r'$\dot{x}(t_*)$')
ax2.set_xlabel(r'$C$')
ax2.set_ylabel(r'$\dot{x}(t_*)$')

fig.legend(loc='upper left', bbox_to_anchor=(0.14, 0.80))
# fig.tight_layout()
fig.suptitle(r'$x_{k-1}$ and excessive velocity $\dot{x}(t_*)$ over $C$ at $x_0=$' f' {start_loc}.' '\n' r'Left wall at $x~=-1.5$', y=1.1)  # otherwise the right y-label is slightly clipped

### Are two-phase solutions performing better?

*Attention*: Modification necessary: self.min_position = -1.2 in continuous_mountain_car.py

In [ ]:
x0s = np.linspace(-0.6, -0.4, 100)
c1s = db["c1s_linspace_100"]
c2 = db["c2_optimum"]

In [ ]:
single_phase_cs = []
for x in x0s:
    c = find_single_phase_c(x)
    single_phase_cs.append(c)
db["single_phase_cs_linspace_100"] = single_phase_cs

In [ ]:
# Single-phase results
kwargs = []
for x in x0s:
    kwargs.append({'start_loc': x, 'single_phase': True})

single_phase_cs = db["single_phase_cs_linspace_100"]
single_cs = []
for c in single_phase_cs:
    single_cs.append([c, 0.0])

pool = mp.Pool(mp.cpu_count())
allres_single_phase = pool.starmap(parallel.job_analytic_solution_policy, zip(single_cs, kwargs))



In [ ]:
# Two-phase results
kwargs = []
for x in x0s:
    kwargs.append({'start_loc': x, 'single_phase': False})

cs = []
for c in c1s:
    cs.append([c, c2])

pool = mp.Pool(mp.cpu_count())
allres_two_phase = pool.starmap(parallel.job_analytic_solution_policy, zip(cs, kwargs))

In [ ]:
r_single = [a[1] for a in allres_single_phase]
r_two = [a[1] for a in allres_two_phase]

In [ ]:
fig, ax1 = plt.subplots()
fig.set_figwidth(9)

ax1.plot(x0s, r_single, '.', label='single-phase')
ax1.plot(x0s, r_two, '.', label='two-phase')
ax1.set_xlabel(r'$x_0$')
ax1.set_ylabel(r'$R$')
ax1.legend(loc='best')

#fig.legend(loc='lower center', bbox_to_anchor=(0.17, 0.68))
fig.tight_layout()
fig.suptitle(r'Analytic solution: Loss $\ell$ over different $x_0$' , y=1.1)  # otherwise the right y-label is slightly clipped

## Compare policies, $x_0 = -0.6$

In [ ]:
x0 = -0.6
res = run_analytic_policy(c1=db["c1_optimum"], c2 = db["c2_optimum"], start=x0)

db["analytic_episode_reward"] = res[0]
db['analytic_v_target'] = res[1]
db['analytic_episode_len'] = res[2]

In [ ]:
ars_episode_reward = db["ars_episode_reward"]
ddpg_episode_reward = db["ddpg_episode_reward"]
a2c_episode_reward = db["a2c_episode_reward"]
trpo_episode_reward = db["trpo_episode_reward"]
td3_episode_reward = db["td3_episode_reward"]
sac_episode_reward = db["sac_episode_reward"]
tqc_episode_reward = db["tqc_episode_reward"]
ppo_episode_reward = db["ppo_episode_reward"]

ars_episode_len = db["ars_episode_len"]
ddpg_episode_len = db["ddpg_episode_len"]
a2c_episode_len = db["a2c_episode_len"]
trpo_episode_len = db["trpo_episode_len"]
td3_episode_len = db["td3_episode_len"]
sac_episode_len = db["sac_episode_len"]
tqc_episode_len = db["tqc_episode_len"]
ppo_episode_len = db["ppo_episode_len"]

ars_v_target = db["ars_v_target"]
ddpg_v_target = db["ddpg_v_target"]
a2c_v_target = db["a2c_v_target"]
trpo_v_target = db["trpo_v_target"]
td3_v_target = db["td3_v_target"]
sac_v_target = db["sac_v_target"]
tqc_v_target = db["tqc_v_target"]
ppo_v_target = db["ppo_v_target"]

analytic_episode_reward = db["analytic_episode_reward"]
analytic_episode_len = db['analytic_episode_len']
analytic_v_target = db['analytic_v_target_smallest_delta']

chebyshev_episode_reward = db["chebyshev_episode_reward"]
chebyshev_episode_len = db["chebyshev_episode_len"]
chebyshev_v_target = db["chebyshev_v_target"]

chebyshev_episode_reward_deg5 = db["chebyshev_episode_reward_deg5"]
chebyshev_episode_len_deg5 = db["chebyshev_episode_len_deg5"]
chebyshev_v_target_deg5 = db["chebyshev_v_target_deg5"]

data = {'ars': ars_episode_reward, 'ddpg': ddpg_episode_reward, 'a2c': a2c_episode_reward, 'trpo': trpo_episode_reward, 'td3': td3_episode_reward, 'sac': sac_episode_reward, 'tqc': tqc_episode_reward, 'ppo': ppo_episode_reward, 'analytic': analytic_episode_reward, 'chebyshev': chebyshev_episode_reward}
sorted_data = dict(sorted(data.items(), key=lambda item: item[1], reverse=True))
len_data = {'ars': ars_episode_len, 'ddpg': ddpg_episode_len, 'a2c': a2c_episode_len, 'trpo': trpo_episode_len, 'td3': td3_episode_len, 'sac': sac_episode_len, 'tqc': tqc_episode_len, 'ppo': ppo_episode_len, 'analytic': analytic_episode_len, 'chebyshev': chebyshev_episode_len}
sorted_len_data = {key: len_data[key] for key in sorted_data if key in len_data}
v_data = {'ars': ars_v_target, 'ddpg': ddpg_v_target, 'a2c': a2c_v_target, 'trpo': trpo_v_target, 'td3': td3_v_target, 'sac': sac_v_target, 'tqc': tqc_v_target, 'ppo': ppo_v_target, 'analytic': analytic_v_target, 'chebyshev': chebyshev_v_target}
sorted_v_data = {key: v_data[key] for key in sorted_data if key in v_data}

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
fig.set_figwidth(24)

# Plt1: Extract names and values
names = list(sorted_data.keys())
values = list(sorted_data.values())

# Create a consistent color mapping for names
unique_names = list(sorted_data.keys())  # Start with names from data1
colors = [plt.cm.tab10(i % 10) for i in range(len(unique_names))]
color_map = {name: colors[i] for i, name in enumerate(unique_names)}
color_map[r'analytic, $\Delta \uparrow$'] = color_map['analytic']
color_map[r'analytic, $\Delta \downarrow$'] = color_map['analytic']

# Include new names from the second dictionary and assign new colors if needed
for name in len_data.keys():
    if name not in color_map:
        color_map[name] = plt.cm.tab10(len(color_map) % 10)  # Cycle through colors

for i, (name, value) in enumerate(sorted_data.items()):
    ax1.scatter(i, value, color=color_map[name], label=name)
    # Add the value next to the point
    ax1.text(i, value, f"{value:.2f}", fontsize=10, verticalalignment='bottom')

ax1.set_xticks(range(len(names)), names, fontsize=10, rotation=45)
ax1.set_title("Episode Reward")
ax1.set_ylim([90,100.5])


# Plot2
names = list(sorted_len_data.keys())
values = list(sorted_len_data.values())

for i, (name, value) in enumerate(sorted_len_data.items()):
    ax2.scatter(i, value, color=color_map[name], label=name)
    # Add the value next to the point
    ax2.text(i, value, f"{value}", fontsize=10, verticalalignment='bottom')

# Set the names as x-axis ticks
ax2.set_xticks(range(len(names)), names, fontsize=10, rotation=45)

# Customize the plot
ax2.set_title("Episode Length")


# Plot3
names = list(sorted_v_data.keys())
values = list(sorted_v_data.values())

for i, (name, value) in enumerate(sorted_v_data.items()):
    ax3.scatter(i, value, color=color_map[name], label=name)
    # Add the value next to the point
    ax3.text(i, value, f"{value:.5f}", fontsize=10, verticalalignment='bottom')

# Set the names as x-axis ticks
ax3.set_xticks(range(len(names)), names, fontsize=10, rotation=45)

# Customize the plot
ax3.set_title("Target Velocity")

fig.tight_layout()  # otherwise the right y-label is slightly clipped
fig.suptitle(r'Reward, episode length and velocity at target for different trained agents' '\n' r'Left wall at $x=-1.2$, Starting at $x_0=-0.6$' '\n' r'$C_1=$' f'{db["c1_optimum"]:.4f}, ' r'$C_2=$' f'{db["c2_optimum"]:.4f}', y=1.2)  # otherwise the right y-label is slightly clipped

In [ ]:
fig = plt.figure(figsize=(10, 8))
gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1])

ax1 = fig.add_subplot(gs[0, :])  # Span all columns
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

# Plt1: Extract names and values
names = list(sorted_data.keys())
values = list(sorted_data.values())

# Create a consistent color mapping for names
unique_names = list(sorted_data.keys())  # Start with names from data1
colors = [plt.cm.tab10(i % 10) for i in range(len(unique_names))]
color_map = {name: colors[i] for i, name in enumerate(unique_names)}
color_map[r'analytic, $\Delta \uparrow$'] = color_map['analytic']
color_map[r'analytic, $\Delta \downarrow$'] = color_map['analytic']

# Include new names from the second dictionary and assign new colors if needed
for name in len_data.keys():
    if name not in color_map:
        color_map[name] = plt.cm.tab10(len(color_map) % 10)  # Cycle through colors

for i, (name, value) in enumerate(sorted_data.items()):
    ax1.scatter(i, value, color=color_map[name], label=name)
    # Add the value next to the point
    ax1.text(i, value, f"{value:.2f}", fontsize=10, verticalalignment='bottom')

ax1.set_xticks(range(len(names)), names, fontsize=10, rotation=45)
ax1.set_title("Episode Reward")
ax1.set_ylim([90,100.5])


# Plot2
names = list(sorted_len_data.keys())
values = list(sorted_len_data.values())

for i, (name, value) in enumerate(sorted_len_data.items()):
    ax2.scatter(i, value, color=color_map[name], label=name)
    # Add the value next to the point
    ax2.text(i, value, f"{value}", fontsize=10, verticalalignment='bottom')

# Set the names as x-axis ticks
ax2.set_xticks(range(len(names)), names, fontsize=10, rotation=45)

# Customize the plot
ax2.set_title("Episode Length")


# Plot3
names = list(sorted_v_data.keys())
values = list(sorted_v_data.values())

for i, (name, value) in enumerate(sorted_v_data.items()):
    ax3.scatter(i, value, color=color_map[name], label=name)
    # Add the value next to the point
    ax3.text(i, value, f"{value:.5f}", fontsize=10, verticalalignment='bottom')

# Set the names as x-axis ticks
ax3.set_xticks(range(len(names)), names, fontsize=10, rotation=45)

# Customize the plot
ax3.set_title("Target Velocity")

fig.tight_layout()  # otherwise the right y-label is slightly clipped
fig.suptitle(r'Reward, episode length and velocity at target for different trained agents' '\n' r'Left wall at $x=-1.2$, Starting at $x_0=-0.6$' '\n' r'$C_1=$' f'{db["c1_optimum"]:.4f}, ' r'$C_2=$' f'{db["c2_optimum"]:.4f}', y=1.2)  # otherwise the right y-label is slightly clipped

## Comparing reward over different $x_0$

We compare policies against the fixed "worst-case C" policy determined at $x_0=-\frac{\pi}{6}$.   

In [ ]:
algos = ['ppo', 'tqc', 'sac', 'td3', 'trpo', 'a2c', 'ddpg', 'ars']
xs = np.linspace(-0.6, -0.4, 100)

In [ ]:
results = []

for a in algos:
    kwargs = {'algo': a}
    pool = mp.Pool(mp.cpu_count())
    results.append(pool.starmap(parallel.job_get_episode_reward_rl_zoo3_model, zip(xs, repeat(kwargs))))

db['huggingface_single_episode_results'] = results

In [ ]:
ars_episode_reward

In [ ]:
kwargs = []
for x in xs:
    kwargs.append({'start_loc': x, 'cin_mode': 'absolute'})

cin = [db["c1_max_1000"], db["c2_optimum"]]

pool = mp.Pool(mp.cpu_count())
allres_x_analytic = pool.starmap(parallel.job_analytic_solution_policy, zip(repeat(cin), kwargs))

db['analytic_single_episode_results'] = allres_x_analytic

In [ ]:
kwargs = []
for x in xs:
    kwargs.append({'start_loc': x})

coeffs = db['chebyshev_reinforce_best_single_episode_result_coeffs']

pool = mp.Pool(mp.cpu_count())
allres_x_chebyshev3 = pool.starmap(parallel.job_get_episode_reward, zip(repeat(coeffs), kwargs))

db['chebyshev_single_episode_results'] = allres_x_chebyshev3

In [ ]:
kwargs = []
for x in xs:
    kwargs.append({'start_loc': x, 'degree': 5})

coeffs = db['chebyshev_deg5_reinforce_best_single_episode_result_coeffs']

pool = mp.Pool(mp.cpu_count())
allres_x_chebyshev5 = pool.starmap(parallel.job_get_episode_reward, zip(repeat(coeffs), kwargs))

db['chebyshev_deg5_single_episode_results'] = allres_x_chebyshev5

In [ ]:
from utils import exp_run

coeffs = db['mountaincar_ch_ars_best_agent_params']
algo = 'ars'
single_episode_results_chebyshev_ars = []

for x in xs:
    model, eval_env = exp_run.get_sb3_polynomial_model_and_eval_env(basis='chebyshev', coeffs=coeffs, algo=algo)
    single_episode_results_chebyshev_ars.append(exp_run.run_sb3_model(model, eval_env, options={'low': x, 'high': x}, return_observations=True))

coeffs = db['mountaincar_ch_ppo_best_agent_params']
algo = 'ppo'
single_episode_results_chebyshev_ppo = []

for x in xs:
    model, eval_env = exp_run.get_sb3_polynomial_model_and_eval_env(basis='chebyshev', coeffs=coeffs, algo=algo)
    single_episode_results_chebyshev_ppo.append(exp_run.run_sb3_model(model, eval_env, options={'low': x, 'high': x}, return_observations=True))

db['chebyshev_single_episode_results_ars'] = single_episode_results_chebyshev_ars
db['chebyshev_single_episode_results_ppo'] = single_episode_results_chebyshev_ppo

In [ ]:
results = db['huggingface_single_episode_results']
results_analytic = db['analytic_single_episode_results']
results_chebyshev3= db['chebyshev_single_episode_results'] 
# results_chebyshev5= db['chebyshev_deg5_single_episode_results'] 

# results_chebyshev_ars = db['mountaincar_chebyshev_ars_eval_results_20251105']
# results_chebyshev_ppo = db['mountaincar_chebyshev_ppo_eval_results_20251220']
single_episode_results_chebyshev_ars = db['chebyshev_single_episode_results_ars']
single_episode_results_chebyshev_ppo = db['chebyshev_single_episode_results_ppo']

r_analytic = [a[1] for a in results_analytic]
target_velocities_analytic = [a[3] for a in results_analytic]
steps_analytic = [len(a[4]) for a in results_analytic]

r_chebyshev3 = [a[0] for a in results_chebyshev3]
target_velocities_chebyshev3 = [a[2][-1][1] for a in results_chebyshev3]
steps_chebyshev3 = [len(a[2]) for a in results_chebyshev3]

# r_chebyshev5 = [a[0] for a in results_chebyshev5]
# target_velocities_chebyshev5 = [a[2][-1][1] for a in results_chebyshev5]
# steps_chebyshev5 = [len(a[2]) for a in results_chebyshev5]

r_chebyshev3_ars = [a[0][0][0] for a in single_episode_results_chebyshev_ars]
target_velocities_chebyshev3_ars = [denormalize(a[1][-1][0][-1], 0.07, -0.07, 1.0, -1.0) for a in single_episode_results_chebyshev_ars]
steps_chebyshev3_ars = [len(a[1]) for a in single_episode_results_chebyshev_ars]

r_chebyshev3_ppo = [a[0][0][0] for a in single_episode_results_chebyshev_ppo]
target_velocities_chebyshev3_ppo= [denormalize(a[1][-1][0][-1], 0.07, -0.07, 1.0, -1.0) for a in single_episode_results_chebyshev_ppo]
steps_chebyshev3_ppo = [len(a[1]) for a in single_episode_results_chebyshev_ppo]

chebyshev3_ars_reward_std = np.std([r for r in r_chebyshev3_ars])
chebyshev3_ppo_reward_std = np.std([r for r in r_chebyshev3_ppo])
chebyshev3_reward_std = np.std([r for r in r_chebyshev3])
analytic_reward_std = np.std([r for r in r_analytic])
ars_reward_std = np.std([r[0] for r in results[7]])
ddpg_reward_std = np.std([r[0] for r in results[6]])
a2c_reward_std = np.std([r[0] for r in results[5]])
trpo_reward_std = np.std([r[0] for r in results[4]])
td3_reward_std = np.std([r[0] for r in results[3]])
sac_reward_std = np.std([r[0] for r in results[2]])
tqc_reward_std = np.std([r[0] for r in results[1]])
ppo_reward_std = np.std([r[0] for r in results[0]])

chebyshev3_ars_episode_len_std = np.std([r for r in steps_chebyshev3_ars])
chebyshev3_ppo_episode_len_std = np.std([r for r in steps_chebyshev3_ppo])
chebyshev3_episode_len_std = np.std([r for r in steps_chebyshev3])
analytic_episode_len_std = np.std([r for r in steps_analytic])
ars_episode_len_std = np.std([r[1] for r in results[7]])
ddpg_episode_len_std = np.std([r[1] for r in results[6]])
a2c_episode_len_std = np.std([r[1] for r in results[5]])
trpo_episode_len_std = np.std([r[1] for r in results[4]])
td3_episode_len_std = np.std([r[1] for r in results[3]])
sac_episode_len_std = np.std([r[1] for r in results[2]])
tqc_episode_len_std = np.std([r[1] for r in results[1]])
ppo_episode_len_std = np.std([r[1] for r in results[0]])

chebyshev3_ars_v_target_std = np.std([r for r in target_velocities_chebyshev3_ars])
chebyshev3_ppo_v_target_std = np.std([r for r in target_velocities_chebyshev3_ppo])
chebyshev3_v_target_std = np.std([r for r in target_velocities_chebyshev3])
analytic_v_target_std = np.std([r for r in target_velocities_analytic])
ars_v_target_std = np.std([r[2] for r in results[7]])
ddpg_v_target_std = np.std([r[2] for r in results[6]])
a2c_v_target_std = np.std([r[2] for r in results[5]])
trpo_v_target_std = np.std([r[2] for r in results[4]])
td3_v_target_std = np.std([r[2] for r in results[3]])
sac_v_target_std = np.std([r[2] for r in results[2]])
tqc_v_target_std = np.std([r[2] for r in results[1]])
ppo_v_target_std = np.std([r[2] for r in results[0]])

In [ ]:
def print_block(title, data, precision=4):
    print(f"\n{'=' * len(title)}")
    print(title)
    print(f"{'=' * len(title)}")
    for name, value in data:
        print(f"{name:<25}: {value:.{precision}f}")

In [ ]:
print_block(
    "Reward Standard Deviation",
    [
        ("Chebyshev3 + ARS", chebyshev3_ars_reward_std),
        ("Chebyshev3 + PPO", chebyshev3_ppo_reward_std),
        ("Chebyshev3", chebyshev3_reward_std),
        ("Analytic", analytic_reward_std),
        ("ARS", ars_reward_std),
        ("DDPG", ddpg_reward_std),
        ("A2C", a2c_reward_std),
        ("TRPO", trpo_reward_std),
        ("TD3", td3_reward_std),
        ("SAC", sac_reward_std),
        ("TQC", tqc_reward_std),
        ("PPO", ppo_reward_std),
    ]
)


# -------- Episode Length STD --------
print_block(
    "Episode Length Standard Deviation",
    [
        ("Chebyshev3 + ARS", chebyshev3_ars_episode_len_std),
        ("Chebyshev3 + PPO", chebyshev3_ppo_episode_len_std),
        ("Chebyshev3", chebyshev3_episode_len_std),
        ("Analytic", analytic_episode_len_std),
        ("ARS", ars_episode_len_std),
        ("DDPG", ddpg_episode_len_std),
        ("A2C", a2c_episode_len_std),
        ("TRPO", trpo_episode_len_std),
        ("TD3", td3_episode_len_std),
        ("SAC", sac_episode_len_std),
        ("TQC", tqc_episode_len_std),
        ("PPO", ppo_episode_len_std),
    ]
)


# -------- Target Velocity STD --------
print_block(
    "Target Velocity Standard Deviation",
    [
        ("Chebyshev3 + ARS", chebyshev3_ars_v_target_std),
        ("Chebyshev3 + PPO", chebyshev3_ppo_v_target_std),
        ("Chebyshev3", chebyshev3_v_target_std),
        ("Analytic", analytic_v_target_std),
        ("ARS", ars_v_target_std),
        ("DDPG", ddpg_v_target_std),
        ("A2C", a2c_v_target_std),
        ("TRPO", trpo_v_target_std),
        ("TD3", td3_v_target_std),
        ("SAC", sac_v_target_std),
        ("TQC", tqc_v_target_std),
        ("PPO", ppo_v_target_std),
    ]
)


In [ ]:
results = db['huggingface_single_episode_results']
results_analytic = db['analytic_single_episode_results']
results_chebyshev3= db['chebyshev_single_episode_results'] 
# results_chebyshev5= db['chebyshev_deg5_single_episode_results'] 

# results_chebyshev_ars = db['mountaincar_chebyshev_ars_eval_results_20251105']
# results_chebyshev_ppo = db['mountaincar_chebyshev_ppo_eval_results_20251220']
single_episode_results_chebyshev_ars = db['chebyshev_single_episode_results_ars']
single_episode_results_chebyshev_ppo = db['chebyshev_single_episode_results_ppo']

r_analytic = [a[1] for a in results_analytic]
target_velocities_analytic = [a[3] for a in results_analytic]
steps_analytic = [len(a[4]) for a in results_analytic]

r_chebyshev3 = [a[0] for a in results_chebyshev3]
target_velocities_chebyshev3 = [a[2][-1][1] for a in results_chebyshev3]
steps_chebyshev3 = [len(a[2]) for a in results_chebyshev3]

# r_chebyshev5 = [a[0] for a in results_chebyshev5]
# target_velocities_chebyshev5 = [a[2][-1][1] for a in results_chebyshev5]
# steps_chebyshev5 = [len(a[2]) for a in results_chebyshev5]

r_chebyshev3_ars = [a[0][0][0] for a in single_episode_results_chebyshev_ars]
target_velocities_chebyshev3_ars = [denormalize(a[1][-1][0][-1], 0.07, -0.07, 1.0, -1.0) for a in single_episode_results_chebyshev_ars]
steps_chebyshev3_ars = [len(a[1]) for a in single_episode_results_chebyshev_ars]

r_chebyshev3_ppo = [a[0][0][0] for a in single_episode_results_chebyshev_ppo]
target_velocities_chebyshev3_ppo= [denormalize(a[1][-1][0][-1], 0.07, -0.07, 1.0, -1.0) for a in single_episode_results_chebyshev_ppo]
steps_chebyshev3_ppo = [len(a[1]) for a in single_episode_results_chebyshev_ppo]

#chebyshev5_episode_reward = [np.min([r for r in r_chebyshev5]), np.mean([r for r in r_chebyshev5]), np.max([r for r in r_chebyshev5])]
chebyshev3_ars_episode_reward = [np.min([r for r in r_chebyshev3_ars]), np.mean([r for r in r_chebyshev3_ars]), np.max([r for r in r_chebyshev3_ars])]
chebyshev3_ppo_episode_reward = [np.min([r for r in r_chebyshev3_ppo]), np.mean([r for r in r_chebyshev3_ppo]), np.max([r for r in r_chebyshev3_ppo])]
chebyshev3_episode_reward = [np.min([r for r in r_chebyshev3]), np.mean([r for r in r_chebyshev3]), np.max([r for r in r_chebyshev3])]
analytic_episode_reward = [np.min([r for r in r_analytic]), np.mean([r for r in r_analytic]), np.max([r for r in r_analytic])]
ars_episode_reward = [np.min([r[0] for r in results[7]]), np.mean([r[0] for r in results[7]]), np.max([r[0] for r in results[7]])]
ddpg_episode_reward = [np.min([r[0] for r in results[6]]), np.mean([r[0] for r in results[6]]), np.max([r[0] for r in results[6]])]
a2c_episode_reward = [np.min([r[0] for r in results[5]]), np.mean([r[0] for r in results[5]]), np.max([r[0] for r in results[5]])]
trpo_episode_reward = [np.min([r[0] for r in results[4]]), np.mean([r[0] for r in results[4]]), np.max([r[0] for r in results[4]])]
td3_episode_reward = [np.min([r[0] for r in results[3]]), np.mean([r[0] for r in results[3]]), np.max([r[0] for r in results[3]])]
sac_episode_reward = [np.min([r[0] for r in results[2]]), np.mean([r[0] for r in results[2]]), np.max([r[0] for r in results[2]])]
tqc_episode_reward = [np.min([r[0] for r in results[1]]), np.mean([r[0] for r in results[1]]), np.max([r[0] for r in results[1]])]
ppo_episode_reward = [np.min([r[0] for r in results[0]]), np.mean([r[0] for r in results[0]]), np.max([r[0] for r in results[0]])]

#chebyshev5_episode_len = [np.min([r for r in steps_chebyshev5]), np.mean([r for r in steps_chebyshev5]), np.max([r for r in steps_chebyshev5])]
chebyshev3_ars_episode_len = [np.min([r for r in steps_chebyshev3_ars]), np.mean([r for r in steps_chebyshev3_ars]), np.max([r for r in steps_chebyshev3_ars])]
chebyshev3_ppo_episode_len = [np.min([r for r in steps_chebyshev3_ppo]), np.mean([r for r in steps_chebyshev3_ppo]), np.max([r for r in steps_chebyshev3_ppo])]
chebyshev3_episode_len = [np.min([r for r in steps_chebyshev3]), np.mean([r for r in steps_chebyshev3]), np.max([r for r in steps_chebyshev3])]
analytic_episode_len = [np.min([r for r in steps_analytic]), np.mean([r for r in steps_analytic]), np.max([r for r in steps_analytic])]
ars_episode_len = [np.min([r[1] for r in results[7]]), np.mean([r[1] for r in results[7]]), np.max([r[1] for r in results[7]])]
ddpg_episode_len = [np.min([r[1] for r in results[6]]), np.mean([r[1] for r in results[6]]), np.max([r[1] for r in results[6]])]
a2c_episode_len = [np.min([r[1] for r in results[5]]), np.mean([r[1] for r in results[5]]), np.max([r[1] for r in results[5]])]
trpo_episode_len = [np.min([r[1] for r in results[4]]), np.mean([r[1] for r in results[4]]), np.max([r[1] for r in results[4]])]
td3_episode_len = [np.min([r[1] for r in results[3]]), np.mean([r[1] for r in results[3]]), np.max([r[1] for r in results[3]])]
sac_episode_len = [np.min([r[1] for r in results[2]]), np.mean([r[1] for r in results[2]]), np.max([r[1] for r in results[2]])]
tqc_episode_len = [np.min([r[1] for r in results[1]]), np.mean([r[1] for r in results[1]]), np.max([r[1] for r in results[1]])]
ppo_episode_len = [np.min([r[1] for r in results[0]]), np.mean([r[1] for r in results[0]]), np.max([r[1] for r in results[0]])]

#chebyshev5_v_target = [np.min([r for r in target_velocities_chebyshev5]), np.mean([r for r in target_velocities_chebyshev5]), np.max([r for r in target_velocities_chebyshev5])]
chebyshev3_ars_v_target = [np.min([r for r in target_velocities_chebyshev3_ars]), np.mean([r for r in target_velocities_chebyshev3_ars]), np.max([r for r in target_velocities_chebyshev3_ars])]
chebyshev3_ppo_v_target = [np.min([r for r in target_velocities_chebyshev3_ppo]), np.mean([r for r in target_velocities_chebyshev3_ppo]), np.max([r for r in target_velocities_chebyshev3_ppo])]
chebyshev3_v_target = [np.min([r for r in target_velocities_chebyshev3]), np.mean([r for r in target_velocities_chebyshev3]), np.max([r for r in target_velocities_chebyshev3])]
analytic_v_target = [np.min([r for r in target_velocities_analytic]), np.mean([r for r in target_velocities_analytic]), np.max([r for r in target_velocities_analytic])]
ars_v_target = [np.min([r[2] for r in results[7]]), np.mean([r[2] for r in results[7]]), np.max([r[2] for r in results[7]])]
ddpg_v_target = [np.min([r[2] for r in results[6]]), np.mean([r[2] for r in results[6]]), np.max([r[2] for r in results[6]])]
a2c_v_target = [np.min([r[2] for r in results[5]]), np.mean([r[2] for r in results[5]]), np.max([r[2] for r in results[5]])]
trpo_v_target = [np.min([r[2] for r in results[4]]), np.mean([r[2] for r in results[4]]), np.max([r[2] for r in results[4]])]
td3_v_target = [np.min([r[2] for r in results[3]]), np.mean([r[2] for r in results[3]]), np.max([r[2] for r in results[3]])]
sac_v_target = [np.min([r[2] for r in results[2]]), np.mean([r[2] for r in results[2]]), np.max([r[2] for r in results[2]])]
tqc_v_target = [np.min([r[2] for r in results[1]]), np.mean([r[2] for r in results[1]]), np.max([r[2] for r in results[1]])]
ppo_v_target = [np.min([r[2] for r in results[0]]), np.mean([r[2] for r in results[0]]), np.max([r[2] for r in results[0]])]

In [ ]:
fig = plt.figure(figsize=(10, 8))
gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1])

ax1 = fig.add_subplot(gs[0, :])  # Span all columns
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

# Plt1: Extract names and values
names = list(sorted_data.keys())
values = list(sorted_data.values())

# Create a consistent color mapping for names
unique_names = list(sorted_data.keys())  # Start with names from data1
colors = [plt.cm.tab10(i % 10) for i in range(len(unique_names))]
color_map = {name: colors[i] for i, name in enumerate(unique_names)}

# # Include new names from the second dictionary and assign new colors if needed
# for name in len_data.keys():
#     if name not in color_map:
#         color_map[name] = plt.cm.tab10(len(color_map) % 10)  # Cycle through colors

for i, (name, value) in enumerate(sorted_data.items()):
    ax1.scatter(i, value[0], color=color_map[name], label=name, marker='v')
    ax1.scatter(i, value[1], color=color_map[name], label=name, marker='o')
    ax1.scatter(i, value[2], color=color_map[name], label=name, marker='^')
    # Add the value next to the point
    if value[1] > 90:
        #ax1.text(i, value[0], f"{value[0]:.2f}", fontsize=10, verticalalignment='bottom')
        ax1.text(i, value[1], f"{value[1]:.2f}", fontsize=10, verticalalignment='bottom')
        #ax1.text(i, value[2], f"{value[2]:.2f}", fontsize=10, verticalalignment='bottom')

ax1.set_xticks(range(len(names)), names, fontsize=10, rotation=45)
ax1.set_title("Episode Reward")
ax1.set_ylim([90,100])


# Plot2
names = list(sorted_len_data.keys())
values = list(sorted_len_data.values())

for i, (name, value) in enumerate(sorted_len_data.items()):
    ax2.scatter(i, value[0], color=color_map[name], label=name, marker='v')
    ax2.scatter(i, value[1], color=color_map[name], label=name, marker='o')
    ax2.scatter(i, value[2], color=color_map[name], label=name, marker='^')
    # Add the value next to the point
    #ax2.text(i, value[0], f"{value[0]:.2f}", fontsize=10, verticalalignment='bottom')
    ax2.text(i, value[1], f"{value[1]:.2f}", fontsize=10, verticalalignment='bottom')
    #ax2.text(i, value[2], f"{value[2]:.2f}", fontsize=10, verticalalignment='bottom')

# Set the names as x-axis ticks
ax2.set_xticks(range(len(names)), names, fontsize=10, rotation=45)
ax2.set_title("Episode Length")


# # Plot3
names = list(sorted_v_data.keys())
values = list(sorted_v_data.values())

for i, (name, value) in enumerate(sorted_v_data.items()):
    ax3.scatter(i, value[0], color=color_map[name], label=name, marker='v')
    ax3.scatter(i, value[1], color=color_map[name], label=name, marker='o')
    ax3.scatter(i, value[2], color=color_map[name], label=name, marker='^')
    # Add the value next to the point
    #ax3.text(i, value[0], f"{value[0]:.5f}", fontsize=10, verticalalignment='bottom')
    ax3.text(i, value[1], f"{value[1]:.5f}", fontsize=10, verticalalignment='bottom')
    #ax3.text(i, value[2], f"{value[2]:.5f}", fontsize=10, verticalalignment='bottom')

# Set the names as x-axis ticks
ax3.set_xticks(range(len(names)), names, fontsize=10, rotation=45)
ax3.set_title("Target Velocity")
ax3.set_ylim(0, 0.07)

fig.tight_layout()  # otherwise the right y-label is slightly clipped
fig.suptitle(r'Min, mean and max reward, episode length and velocity at target for different trained agents' '\n' r'Left wall at $x=-1.2$, $100$ evenly spaced starting positions over the interval $x_0 \in [-0.6, -0.4]$' '\n'  r'$C_1=$' f'{db["c1_max_1000"]:.4f}, ' r'$C_2=$' f'{db["c2_optimum"]:.4f}', y=1.2)

## Visualize distance of policies to analytic solution

d(π₁, π₂) = ‖π₁ - π₂‖ = sup_x (π₁ - π₂)(x) or √∫(π₁ - π₂)(x)²
i.e. supremum or l2 norms.

In [ ]:
from scipy.integrate import quad

def supremum_distance_from_arrays(f1_values, f2_values):
    """
    Computes the supremum distance between two arrays of function values.
    
    Parameters:
    - f1_values: Array of f1(x) values at certain x points.
    - f2_values: Array of f2(x) values at the same x points.
    
    Returns:
    - Supremum distance ‖f1 - f2‖∞
    """
    differences = np.abs(f1_values - f2_values)
    return np.max(differences)


def l2_distance_from_arrays(f1_values, f2_values, x_points, y_points):
    """
    Computes the L2 distance between two 2D arrays of function values.
    
    Parameters:
    - f1_values: 2D array of f1(x, y) values on a grid.
    - f2_values: 2D array of f2(x, y) values on the same grid.
    - x_points: 1D array of x coordinates for the grid.
    - y_points: 1D array of y coordinates for the grid.
    
    Returns:
    - L2 distance √(∫∫ (f1(x, y) - f2(x, y))² dx dy)
    """
    dx = x_points[-1]-x_points[0]
    dy = y_points[-1]-y_points[0]
    differences_squared = (f1_values - f2_values) ** 2
    integral = np.sum(differences_squared) * dx * dy/(len(x_points)*len(y_points))
    return np.sqrt(integral)


def evaluate_poly_mrp(mrp, num_points, unnormalized_upper=[0.6, 0.07], unnormalized_lower=[-1.2, -0.07]):
    x = np.linspace(unnormalized_lower[0], unnormalized_upper[0], num_points)
    y = np.linspace(unnormalized_lower[1], unnormalized_upper[1], num_points)

    X, Y = np.meshgrid(x, y)
    states = np.vstack([X.ravel(), Y.ravel()]).T

    # Predict actions for each state
    actions = np.array([mrp.agent.select_action(mrp.normalize(state, max_value=np.array(np.array([0.6, 0.07])), min_value=np.array(np.array([-1.2, -0.07]))))[0] for state in states])

    return [np.array(x), np.array(y), actions]


def evaluate_poly_sb3(algo, coeffs, num_points, unnormalized_upper=[0.6, 0.07], unnormalized_lower=[-1.2, -0.07]):
    x = np.linspace(unnormalized_lower[0], unnormalized_upper[0], num_points)
    y = np.linspace(unnormalized_lower[1], unnormalized_upper[1], num_points)
    X, Y = np.meshgrid(x, y)
    states = np.vstack([X.ravel(), Y.ravel()]).T

    model, _ = exp_run.get_sb3_polynomial_model_and_eval_env(basis='chebyshev', coeffs=coeffs, algo=algo)
    actions = np.array([model.predict(normalize(state, max_value=np.array([0.6, 0.07]), min_value=np.array([-1.2, -0.07])), deterministic=True)[0] for state in states])

    return [np.array(x), np.array(y), actions]


def evaluate_rl_zoo3_model(env, model, num_points, unnormalized_upper=[0.6, 0.07], unnormalized_lower=[-1.2, -0.07]):
    x = np.linspace(unnormalized_lower[0], unnormalized_upper[0], num_points)
    y = np.linspace(unnormalized_lower[1], unnormalized_upper[1], num_points)

    X, Y = np.meshgrid(x, y)
    states = np.vstack([X.ravel(), Y.ravel()]).T

    try:
        if env.norm_obs: 
            normalized = True
        else:
            normalized = False
    except:
        normalized = False

    if not normalized:
        actions = np.array([model.predict(state, deterministic=True)[0] for state in states])
    else:
        actions = np.array([model.predict(env.normalize_obs(state), deterministic=True)[0] for state in states])

    return [np.array(x), np.array(y), actions]


def evaluate_analytic(c1, c2, num_points, unnormalized_upper=[0.6, 0.07], unnormalized_lower=[-1.2, -0.07]):
    x = np.linspace(unnormalized_lower[0], unnormalized_upper[0], num_points)
    y = np.linspace(unnormalized_lower[1], unnormalized_upper[1], num_points)    

    X, Y = np.meshgrid(x, y)
    states = np.vstack([X.ravel(), Y.ravel()]).T

    # Approximate "C2 border trajectory" with a polynomial
    c2_trajectory = parallel.job_analytic_solution_policy([0.0, c2], kwargs={'start_loc': unnormalized_lower[0]})[-1] # generate trajectory
    x_phase2, y_phase2 = zip(*c2_trajectory) # separate the data into x and y lists
    coefficients = np.polyfit(x_phase2, y_phase2, 7) # Fit a 7th degree polynomial
    c2_border_poly = np.poly1d(coefficients)

    # Predict actions for each state
    actions = []
    for state in states:
        if state[1] < c2_border_poly(state[0]):
            c = c1
        else:
            c = c2
        actions.append(get_analytic_action(state, c))

    return [np.array(x), np.array(y), np.array(actions)]

In [ ]:
# # Verify Distance function
# x = np.linspace(2, 11, 10)
# y = np.linspace(4, 6, 10)
# X, Y = np.meshgrid(x, y)

# f = np.sin(X)*np.cos(Y)
# g = np.sin(X)*np.cos(Y)+1

# l2_distance_from_arrays(f,g, x, y)

In [ ]:
db['analytic_res_10000'] = evaluate_analytic(db["c1_pi_ana"], db["c2_optimum"], 100)

In [ ]:
coeffs = db['chebyshev_reinforce_best_single_episode_result_coeffs']
env = gym.make("MountainCarContinuous-v0")

mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3,
                                                            normalize_observations=True,
                                                            mu_coeffs=coeffs)

chebyshev_res = evaluate_poly_mrp(mrp, 100, unnormalized_upper=[0.45, 0.07])
db['chebyshev_evaluation_res_10000'] = chebyshev_res

In [ ]:
coeffs = db['chebyshev_deg5_reinforce_best_single_episode_result_coeffs']
env = gym.make("MountainCarContinuous-v0")

mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=5,
                                                            normalize_observations=True,
                                                            mu_coeffs=coeffs)

chebyshev_deg5_res = evaluate_poly_mrp(mrp, 100, unnormalized_upper=[0.45, 0.07])
db['chebyshev_deg5_evaluation_res_10000'] = chebyshev_deg5_res

In [ ]:
coeffs = db['mountaincar_ch_ppo_best_agent_params']

chebyshev_ppo_res = evaluate_poly_sb3(algo='ppo', coeffs=coeffs, unnormalized_upper=[0.45, 0.07], num_points=100)
db['chebyshev_ppo_evaluation_res_10000'] = chebyshev_ppo_res

In [ ]:
coeffs = db['mountaincar_ch_ars_best_agent_params']

chebyshev_ars_res = evaluate_poly_sb3(algo='ars', coeffs=coeffs, unnormalized_upper=[0.45, 0.07], num_points=100)
db['chebyshev_ars_evaluation_res_10000'] = chebyshev_ars_res

In [ ]:
algos = ['ppo', 'tqc', 'sac', 'td3', 'trpo', 'a2c', 'ddpg', 'ars']

analytic_res = db['analytic_res_10000']
chebyshev_res = db['chebyshev_evaluation_res_10000']
chebyshev_deg5_res = db['chebyshev_deg5_evaluation_res_10000']
chebyshev_ppo_res = db['chebyshev_ppo_evaluation_res_10000']
chebyshev_ars_res = db['chebyshev_ars_evaluation_res_10000']

distances_to_analytic_sup = {}
distances_to_analytic_sup['CH-3'] = supremum_distance_from_arrays(analytic_res[2], chebyshev_res[2])
#distances_to_analytic_sup['chebyshev-5'] = supremum_distance_from_arrays(analytic_res[2], chebyshev_deg5_res[2])
distances_to_analytic_sup['CH-3-ARS'] = supremum_distance_from_arrays(analytic_res[2], chebyshev_ars_res[2])
distances_to_analytic_sup['CH-3-PPO'] = supremum_distance_from_arrays(analytic_res[2], chebyshev_ppo_res[2])

for a in algos:
    model, env = get_rl_zoo3_model_and_generate_env(a)
    res = evaluate_rl_zoo3_model(env, model, 100, unnormalized_upper=[0.45, 0.07])
    distances_to_analytic_sup[a] = supremum_distance_from_arrays(analytic_res[2], [r[0] for r in res[2]])

distances_to_analytic_l2 = {}
distances_to_analytic_l2['CH-3'] = l2_distance_from_arrays(analytic_res[2], chebyshev_res[2], analytic_res[0], analytic_res[1])
#distances_to_analytic_l2['chebyshev-5'] = l2_distance_from_arrays(analytic_res[2], chebyshev_deg5_res[2], analytic_res[0], analytic_res[1])
distances_to_analytic_l2['CH-3-ARS'] = l2_distance_from_arrays(analytic_res[2], chebyshev_ars_res[2].flatten(), analytic_res[0], analytic_res[1])
distances_to_analytic_l2['CH-3-PPO'] = l2_distance_from_arrays(analytic_res[2], chebyshev_ppo_res[2].flatten(), analytic_res[0], analytic_res[1])

for a in algos:
    model, env = get_rl_zoo3_model_and_generate_env(a)
    res = evaluate_rl_zoo3_model(env, model, 100, unnormalized_upper=[0.45, 0.07])
    distances_to_analytic_l2[a] = l2_distance_from_arrays(analytic_res[2], [r[0] for r in res[2]], analytic_res[0], analytic_res[1])

In [ ]:
dict(sorted(distances_to_analytic_sup.items(), key=lambda item: item[1]))

In [ ]:
dict(sorted(distances_to_analytic_l2.items(), key=lambda item: item[1]))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2)
fig.set_figwidth(12)

sorted_distances = dict(sorted(distances_to_analytic_sup.items(), key=lambda item: item[1]))

function_names = list(sorted_distances.keys())
distance_values = list(sorted_distances.values())

ax1.bar(function_names, distance_values, color='cornflowerblue')
ax1.set_xticks(range(len(function_names)), function_names, fontsize=10, rotation=45)
ax1.set_title(r"$‖\pi_{\text{ana}} - \pi‖_\infty$")
ax1.set_ylim(0, 1.5)

for i, value in enumerate(distance_values):
    ax1.text(i, value + 0.02, f'{value:.2f}', ha='center', va='bottom', fontsize=10)

sorted_distances_l2 = dict(sorted(distances_to_analytic_l2.items(), key=lambda item: item[1]))

function_names_l2 = list(sorted_distances_l2.keys())
distance_values_l2 = list(sorted_distances_l2.values())

ax2.bar(function_names_l2, distance_values_l2, color='cornflowerblue')
ax2.set_xticks(range(len(function_names_l2)), function_names_l2, fontsize=10, rotation=45)
ax2.set_title(r"$‖\pi_{\text{ana}} - \pi‖_2$")
ax2.set_ylim(0, 0.5)

for i, value in enumerate(distance_values_l2):
    ax2.text(i, value + 0.001, f'{value:.4f}', ha='center', va='bottom', fontsize=10)

fig.tight_layout()  # otherwise the right y-label is slightly clipped
#fig.suptitle(f'Distance to analytic solution (C1={db["c1_max_1000"]:.4f}, C2={db["c2_optimum"]:.4f}) for different policies' '\n' r'Evaluated at $10000$ evenly spaced data points in the observation space $x \in [-1.2, 0.6]$, $y \in [-0.07, 0.07]$', y=1.2)
fig.savefig(".\\paper-figures\\distance_to_analytic.pdf")


## Are sizes of neural nets too large?

The results above indicate that polynomial agents are superior to agents utilizing NNs.  
Is this because of the approximator type or the net size?  
In SB3, the default net for PPO is an ActorCriticPolicy (MLP) with net_arch = dict(pi=[64, 64], vf=[64, 64]), which is also utilized by our rl-zoo benchmark model.  
First, we reproduce results with training using this net size and then decrease it.   
  
Hyperparameters according to https://huggingface.co/sb3/ppo-MountainCarContinuous-v0 :

```python
OrderedDict([('batch_size', 256),  
             ('clip_range', 0.1),  
             ('ent_coef', 0.00429),  
             ('gae_lambda', 0.9),  
             ('gamma', 0.9999),  
             ('learning_rate', 7.77e-05),  
             ('max_grad_norm', 5),  
             ('n_envs', 1),  
             ('n_epochs', 10),  
             ('n_steps', 8),  
             ('n_timesteps', 20000.0),  
             ('normalize', True),  
             ('policy', 'MlpPolicy'),  
             ('policy_kwargs', 'dict(log_std_init=-3.29, ortho_init=False)'),  
             ('use_sde', True),  
             ('vf_coef', 0.19),  
             ('normalize_kwargs', {'norm_obs': True, 'norm_reward': False})])  ```

In the rl_zoo3 lib directory of our env, we run 
```python
train.py --algo ppo --env MountainCarContinuous-v0 -f logs/
```

and then copy the content of the logs folder to our _rl-zoo-ppo-training_ folder.  
We adapt hyperparameters/ppo.yml for setting hyperparameters to reduce net size accordingly.  
ppo yml settings meeting the Hyperparameters of huggingface:  
```python
MountainCarContinuous-v0:
  normalize: "{'norm_obs': True, 'norm_reward': False}"
  n_envs: 1
  n_timesteps: !!float 20000
  policy: 'MlpPolicy'
  batch_size: 256
  n_steps: 8
  gamma: 0.9999
  learning_rate: !!float 7.77e-05
  ent_coef: 0.00429
  clip_range: 0.1
  n_epochs: 10
  gae_lambda: 0.9
  max_grad_norm: 5
  vf_coef: 0.19
  use_sde: True
  policy_kwargs: "dict(log_std_init=-3.29, ortho_init=False)"
```

In [ ]:
import sys
from rl_zoo3.train import train

In [ ]:
folder="rl-zoo-ppo-training/default-net-size"

sys.argv = ["python", "--algo", "ppo", "--env", "MountainCarContinuous-v0", "-f", folder]

train()

In [ ]:
folder="rl-zoo-ppo-training/64-single-net-size"

sys.argv = ["python", "--algo", "ppo", "--env", "MountainCarContinuous-v0", "-f", folder]

train()

Now we reduce the size to 32, 32 using config:

```python
MountainCarContinuous-v0:
  normalize: "{'norm_obs': True, 'norm_reward': False}"
  n_envs: 1
  n_timesteps: !!float 20000
  policy: 'MlpPolicy'
  batch_size: 256
  n_steps: 8
  gamma: 0.9999
  learning_rate: !!float 7.77e-05
  ent_coef: 0.00429
  clip_range: 0.1
  n_epochs: 10
  gae_lambda: 0.9
  max_grad_norm: 5
  vf_coef: 0.19
  use_sde: True
  policy_kwargs: "dict(log_std_init=-3.29, ortho_init=False, net_arch=dict(pi=[32, 32], vf=[32, 32]))"
```

In [ ]:
folder="rl-zoo-ppo-training/32-net-size"

sys.argv = ["python", "--algo", "ppo", "--env", "MountainCarContinuous-v0", "-f", folder]

train()

In [ ]:
folder="rl-zoo-ppo-training/32-single-net-size"

sys.argv = ["python", "--algo", "ppo", "--env", "MountainCarContinuous-v0", "-f", folder]

train()

Now we reduce the size to 16,16 using config:

```python
MountainCarContinuous-v0:
  normalize: "{'norm_obs': True, 'norm_reward': False}"
  n_envs: 1
  n_timesteps: !!float 20000
  policy: 'MlpPolicy'
  batch_size: 256
  n_steps: 8
  gamma: 0.9999
  learning_rate: !!float 7.77e-05
  ent_coef: 0.00429
  clip_range: 0.1
  n_epochs: 10
  gae_lambda: 0.9
  max_grad_norm: 5
  vf_coef: 0.19
  use_sde: True
  policy_kwargs: "dict(log_std_init=-3.29, ortho_init=False, net_arch=dict(pi=[16,16], vf=[16,16]))"

In [ ]:
folder="rl-zoo-ppo-training/16-net-size"

sys.argv = ["python", "--algo", "ppo", "--env", "MountainCarContinuous-v0", "-f", folder]

train()

Now we reduce the size to 16 (single layer) using config:

```python
MountainCarContinuous-v0:
  normalize: "{'norm_obs': True, 'norm_reward': False}"
  n_envs: 1
  n_timesteps: !!float 20000
  policy: 'MlpPolicy'
  batch_size: 256
  n_steps: 8
  gamma: 0.9999
  learning_rate: !!float 7.77e-05
  ent_coef: 0.00429
  clip_range: 0.1
  n_epochs: 10
  gae_lambda: 0.9
  max_grad_norm: 5
  vf_coef: 0.19
  use_sde: True
  policy_kwargs: "dict(log_std_init=-3.29, ortho_init=False, net_arch=dict(pi=[16], vf=[16]))"

In [ ]:
folder="rl-zoo-ppo-training/16-single-net-size"

sys.argv = ["python", "--algo", "ppo", "--env", "MountainCarContinuous-v0", "-f", folder]

train()

In [ ]:
algos = ['ppo', 'ppo', 'ppo', 'ppo', 'ppo', 'ppo']
folders = ['rl-zoo-ppo-training/default-net-size', 'rl-zoo-ppo-training/64-single-net-size', 'rl-zoo-ppo-training/32-net-size', 'rl-zoo-ppo-training/32-single-net-size', 'rl-zoo-ppo-training/16-net-size', 'rl-zoo-ppo-training/16-single-net-size']
xs = np.linspace(-0.6, -0.4, 20)

results = []

for i in range(len(algos)):
    kwargs = {'algo': algos[i], 'folder': folders[i]}
    pool = mp.Pool(mp.cpu_count())
    results.append(pool.starmap(parallel.job_get_episode_reward_rl_zoo3_model, zip(xs, repeat(kwargs))))

db['ppo_net_size_episode_results'] = results

In [ ]:
results = db['ppo_net_size_episode_results']

ppo6_episode_reward = [np.min([r[0] for r in results[5]]), np.mean([r[0] for r in results[5]]), np.max([r[0] for r in results[5]])]
ppo5_episode_reward = [np.min([r[0] for r in results[4]]), np.mean([r[0] for r in results[4]]), np.max([r[0] for r in results[4]])]
ppo4_episode_reward = [np.min([r[0] for r in results[3]]), np.mean([r[0] for r in results[3]]), np.max([r[0] for r in results[3]])]
ppo3_episode_reward = [np.min([r[0] for r in results[2]]), np.mean([r[0] for r in results[2]]), np.max([r[0] for r in results[2]])]
ppo2_episode_reward = [np.min([r[0] for r in results[1]]), np.mean([r[0] for r in results[1]]), np.max([r[0] for r in results[1]])]
ppo1_episode_reward = [np.min([r[0] for r in results[0]]), np.mean([r[0] for r in results[0]]), np.max([r[0] for r in results[0]])]

ppo6_episode_len = [np.min([r[1] for r in results[5]]), np.mean([r[1] for r in results[5]]), np.max([r[1] for r in results[5]])]
ppo5_episode_len = [np.min([r[1] for r in results[4]]), np.mean([r[1] for r in results[4]]), np.max([r[1] for r in results[4]])]
ppo4_episode_len = [np.min([r[1] for r in results[3]]), np.mean([r[1] for r in results[3]]), np.max([r[1] for r in results[3]])]
ppo3_episode_len = [np.min([r[1] for r in results[2]]), np.mean([r[1] for r in results[2]]), np.max([r[1] for r in results[2]])]
ppo2_episode_len = [np.min([r[1] for r in results[1]]), np.mean([r[1] for r in results[1]]), np.max([r[1] for r in results[1]])]
ppo1_episode_len = [np.min([r[1] for r in results[0]]), np.mean([r[1] for r in results[0]]), np.max([r[1] for r in results[0]])]

ppo6_v_target = [np.min([r[2] for r in results[5]]), np.mean([r[2] for r in results[5]]), np.max([r[2] for r in results[5]])]
ppo5_v_target = [np.min([r[2] for r in results[4]]), np.mean([r[2] for r in results[4]]), np.max([r[2] for r in results[4]])]
ppo4_v_target = [np.min([r[2] for r in results[3]]), np.mean([r[2] for r in results[3]]), np.max([r[2] for r in results[3]])]
ppo3_v_target = [np.min([r[2] for r in results[2]]), np.mean([r[2] for r in results[2]]), np.max([r[2] for r in results[2]])]
ppo2_v_target = [np.min([r[2] for r in results[1]]), np.mean([r[2] for r in results[1]]), np.max([r[2] for r in results[1]])]
ppo1_v_target = [np.min([r[2] for r in results[0]]), np.mean([r[2] for r in results[0]]), np.max([r[2] for r in results[0]])]

data = {'[64,64] (default)': ppo1_episode_reward, '[64]': ppo2_episode_reward, '[32,32]': ppo3_episode_reward, '[32]': ppo4_episode_reward, '[16,16]': ppo5_episode_reward, '[16]': ppo6_episode_reward}
sorted_data = dict(sorted(data.items(), key=lambda item: item[1][1], reverse=True)) # sort by mean reward
len_data = {'[64,64] (default)': ppo1_episode_len, '[64]': ppo2_episode_len, '[32,32]': ppo3_episode_len, '[32]': ppo4_episode_len, '[16,16]': ppo5_episode_len, '[16]': ppo6_episode_len}
sorted_len_data = {key: len_data[key] for key in sorted_data if key in len_data}
v_data = {'[64,64] (default)': ppo1_v_target, '[64]': ppo2_v_target, '[32,32]': ppo3_v_target, '[32]': ppo4_v_target, '[16,16]': ppo5_v_target, '[16]': ppo6_v_target}
sorted_v_data = {key: v_data[key] for key in sorted_data if key in v_data}

In [ ]:
ppo4_episode_reward

In [ ]:
fig = plt.figure(figsize=(10, 8))
gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1])

ax1 = fig.add_subplot(gs[0, :])  # Span all columns
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

# Plt1: Extract names and values
names = list(sorted_data.keys())
values = list(sorted_data.values())

# Create a consistent color mapping for names
unique_names = list(sorted_data.keys())  # Start with names from data1
colors = [plt.cm.tab10(i % 10) for i in range(len(unique_names))]
color_map = {name: colors[i] for i, name in enumerate(unique_names)}

# # Include new names from the second dictionary and assign new colors if needed
# for name in len_data.keys():
#     if name not in color_map:
#         color_map[name] = plt.cm.tab10(len(color_map) % 10)  # Cycle through colors

for i, (name, value) in enumerate(sorted_data.items()):
    ax1.scatter(i, value[0], color=color_map[name], label=name, marker='v')
    ax1.scatter(i, value[1], color=color_map[name], label=name, marker='o')
    ax1.scatter(i, value[2], color=color_map[name], label=name, marker='^')
    # Add the value next to the point
    #ax1.text(i, value[0], f"{value[0]:.2f}", fontsize=10, verticalalignment='bottom')
    ax1.text(i, value[1], f"{value[1]:.2f}", fontsize=10, verticalalignment='bottom')
    #ax1.text(i, value[2], f"{value[2]:.2f}", fontsize=10, verticalalignment='bottom')

ax1.set_xticks(range(len(names)), names, fontsize=10, rotation=45)
ax1.set_title("Episode Reward")
#ax1.set_ylim([90,100])


# Plot2
names = list(sorted_len_data.keys())
values = list(sorted_len_data.values())

for i, (name, value) in enumerate(sorted_len_data.items()):
    ax2.scatter(i, value[0], color=color_map[name], label=name, marker='v')
    ax2.scatter(i, value[1], color=color_map[name], label=name, marker='o')
    ax2.scatter(i, value[2], color=color_map[name], label=name, marker='^')
    # Add the value next to the point
    #ax2.text(i, value[0], f"{value[0]:.2f}", fontsize=10, verticalalignment='bottom')
    ax2.text(i, value[1], f"{value[1]:.2f}", fontsize=10, verticalalignment='bottom')
    #ax2.text(i, value[2], f"{value[2]:.2f}", fontsize=10, verticalalignment='bottom')

# Set the names as x-axis ticks
ax2.set_xticks(range(len(names)), names, fontsize=10, rotation=45)
ax2.set_title("Episode Length")


# # Plot3
names = list(sorted_v_data.keys())
values = list(sorted_v_data.values())

for i, (name, value) in enumerate(sorted_v_data.items()):
    ax3.scatter(i, value[0], color=color_map[name], label=name, marker='v')
    ax3.scatter(i, value[1], color=color_map[name], label=name, marker='o')
    ax3.scatter(i, value[2], color=color_map[name], label=name, marker='^')
    # Add the value next to the point
    #ax3.text(i, value[0], f"{value[0]:.5f}", fontsize=10, verticalalignment='bottom')
    ax3.text(i, value[1], f"{value[1]:.5f}", fontsize=10, verticalalignment='bottom')
    #ax3.text(i, value[2], f"{value[2]:.5f}", fontsize=10, verticalalignment='bottom')

# Set the names as x-axis ticks
ax3.set_xticks(range(len(names)), names, fontsize=10, rotation=45)
ax3.set_title("Target Velocity")
ax3.set_ylim(0, 0.07)

fig.tight_layout()  # otherwise the right y-label is slightly clipped
fig.suptitle(r'Min, mean and max reward, episode length and velocity at target for PPO agents with different MLP "sizes"' '\n' r'Left wall at $x=-1.2$, $20$ evenly spaced starting positions over the interval $x_0 \in [-0.6, -0.4]$', y=1.2)

### Effects of bootstrapping on action

In [ ]:
obs = run_analytic_policy(c1=db['c1_optimum'], c2=db['c2_optimum'], start=-np.pi/6)[-2]

In [ ]:
obs

In [ ]:
plt.plot(obs)

In [ ]:
obs = run_analytic_policy(c1=4.5, c2=db['c2_optimum'], start=-np.pi/6, render=True)[-2]

In [ ]:
plt.plot(obs)

### Additional plots for paper

In [ ]:
start=-0.55

In [ ]:
coeffs = db['chebyshev_reinforce_best_single_episode_result_coeffs']

# Run some actions from trained policy
env = gym.make("MountainCarContinuous-v0", render_mode="human")

mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3,
                                                               normalize_observations=True,
                                                               mu_coeffs=coeffs)

chebyshev_reward = 0.0
chebyshev_observations = []
obs = mrp.reset(options={'low': start, 'high': start})[0]
chebyshev_observations.append(obs)
print(f'Starting at {mrp.unnormalize(chebyshev_observations[0], np.array([0.6, 0.07]), np.array([-1.2, -0.07]))}')
for i in range(1000):
    obs, reward, terminated, truncated, info, action, _ = mrp.step(obs)
    chebyshev_reward += reward
    chebyshev_observations.append(obs)
    #print(reward, r)
    if terminated or truncated:
        break
db['chrei_observations_055'] = chebyshev_observations
db['chrei_reward_055'] = chebyshev_reward

In [ ]:
c2 = db["c2_optimum"]
c1 = db["c1_pi_ana"]
res = run_analytic_policy(c1=c1, c2=c2, start=start)
pi_ana_observations = res[-1]
pi_ana_reward = res[0]
db['pi_ana_observations_055'] = pi_ana_observations
db['pi_ana_reward_055'] = pi_ana_reward

In [ ]:
ars_model, ars_env = get_rl_zoo3_model_and_generate_env('ars')

In [ ]:
ars_reward, _, ars_observations = run_rl_zoo3_model(ars_model, ars_env, start_loc=start)
db['ars_observations_055'] = ars_observations
db['ars_reward_055'] = ars_reward

In [ ]:
ppo_model, ppo_env = get_rl_zoo3_model_and_generate_env('ppo')
ppo_reward, _, ppo_observations = run_rl_zoo3_model(ppo_model, ppo_env, start_loc=start)
db['ppo_observations_055'] = ppo_observations
db['ppo_reward_055'] = ppo_reward

In [ ]:
sac_model, sac_env = get_rl_zoo3_model_and_generate_env('sac')
sac_reward, _, sac_observations = run_rl_zoo3_model(sac_model, sac_env, start_loc=start)
db['sac_observations_055'] = sac_observations
db['sac_reward_055'] = sac_reward

In [ ]:
chars_model, chars_env = exp_run.get_sb3_polynomial_model_and_eval_env(algo='ars', coeffs=db['mountaincar_ch_ars_best_agent_params'])
chars_reward, chars_observations = exp_run.run_sb3_model(chars_model, chars_env, options={'low': start, 'high': start}, return_observations=True)
db['chars_observations_055'] = denormalize_sb3_ch_trajectory(chars_observations)
db['chars_reward_055'] = chars_reward[0][0]

In [ ]:
chppo_model, chppo_env = exp_run.get_sb3_polynomial_model_and_eval_env(algo='ppo', coeffs=db['mountaincar_ch_ppo_best_agent_params'])
chppo_reward, chppo_observations = exp_run.run_sb3_model(chppo_model, chppo_env, options={'low': start, 'high': start}, return_observations=True)
db['chppo_observations_055'] = denormalize_sb3_ch_trajectory(chppo_observations)
db['chppo_reward_055'] = chppo_reward[0][0]

In [ ]:
x0s = np.linspace(-0.6, -0.4, 100)
c1s = db["c1s_linspace_100"]
c2 = db["c2_optimum"]

In [ ]:
ch_ppo_reward_x0_range = []
for x in x0s:
    ch_ppo_reward_x0_range.append(exp_run.run_sb3_model(chppo_model, chppo_env, options={'low': x, 'high': x}, return_observations=False)[0])
db['ch_ppo_reward_x0_range'] = ch_ppo_reward_x0_range

In [ ]:
ch_ppo_reward_x0_range = []
for x in x0s:
    ch_ppo_reward_x0_range.append(exp_run.run_sb3_model(chars_model, chars_env, options={'low': x, 'high': x}, return_observations=False)[0])
db['ch_ars_reward_x0_range'] = ch_ppo_reward_x0_range

In [ ]:
# Chebyshev-3 results
coeffs = db['chebyshev_reinforce_best_single_episode_result_coeffs']

kwargs = []
for x in x0s:
    kwargs.append({'start_loc': x})

pool = mp.Pool(mp.cpu_count())
allres_chebyshev = pool.starmap(parallel.job_get_episode_reward, zip(repeat(coeffs), kwargs))

r_chebyshev = [a[0] for a in allres_chebyshev]
db["chebyshev_reward_x0_range"] = r_chebyshev

## Analytic optimal solution for different x_0

In [ ]:
x0s = np.linspace(-0.6, -0.4, 100)
c1s = db["c1s_linspace_100"]
c2 = db["c2_optimum"]

In [ ]:
# Two-phase results, opt c
kwargs = []
for x in x0s:
    kwargs.append({'start_loc': x, 'single_phase': False})

cs = []
for c in c1s:
    cs.append([c, c2])

pool = mp.Pool(mp.cpu_count())
allres_two_phase = pool.starmap(parallel.job_analytic_solution_policy, zip(cs, kwargs))

r_two = [a[1] for a in allres_two_phase]
v_two = [a[3] for a in allres_two_phase]
len_two = [len(a[4]) for a in allres_two_phase]

db['r_two'] = r_two
db['v_two'] = v_two
db['len_two'] = len_two

In [ ]:
r_two = db['r_two']
print(f'{np.min(r_two)}, {np.std(r_two)} {np.mean(r_two)}, {np.max(r_two)}')

In [ ]:
v_two = db['v_two']
print(f'{np.min(v_two)}, {np.std(v_two)}, {np.mean(v_two)}, {np.max(v_two)}')

In [ ]:
len_two = db['len_two']
print(f'{np.min(len_two)}, {np.std(len_two)}, {np.mean(len_two)}, {np.max(len_two)}')

In [ ]:
# Pi_ana results 
kwargs = []
for x in x0s:
    kwargs.append({'start_loc': x, 'single_phase': False})

c = [db['c1_pi_ana'], db["c2_optimum"]]

pool = mp.Pool(mp.cpu_count())
allres_pi_ana = pool.starmap(parallel.job_analytic_solution_policy, zip(repeat(c), kwargs))

r_pi_ana = [a[1] for a in allres_pi_ana]
db['r_pi_ana'] = r_pi_ana

In [ ]:
# ARS results
kwargs = {'algo': 'ars'}


pool = mp.Pool(mp.cpu_count())
allres_ars = pool.starmap(parallel.job_get_episode_reward_rl_zoo3_model, zip(x0s, repeat(kwargs)))

r_ars = [a[0] for a in allres_ars]
db["ars_reward_x0_range"] = r_ars

In [ ]:
# ARS results
kwargs = {'algo': 'ppo'}


pool = mp.Pool(mp.cpu_count())
allres_ars = pool.starmap(parallel.job_get_episode_reward_rl_zoo3_model, zip(x0s, repeat(kwargs)))

r_ars = [a[0] for a in allres_ars]
db["ppo_reward_x0_range"] = r_ars

In [ ]:
# ARS results
kwargs = {'algo': 'sac'}


pool = mp.Pool(mp.cpu_count())
allres_ars = pool.starmap(parallel.job_get_episode_reward_rl_zoo3_model, zip(x0s, repeat(kwargs)))

r_ars = [a[0] for a in allres_ars]
db["sac_reward_x0_range"] = r_ars

In [ ]:
# ARS results
kwargs = {'algo': 'ddpg'}


pool = mp.Pool(mp.cpu_count())
allres_ars = pool.starmap(parallel.job_get_episode_reward_rl_zoo3_model, zip(x0s, repeat(kwargs)))

r_ars = [a[0] for a in allres_ars]
db["ddpg_reward_x0_range"] = r_ars

In [ ]:
# ARS results
kwargs = {'algo': 'td3'}


pool = mp.Pool(mp.cpu_count())
allres_ars = pool.starmap(parallel.job_get_episode_reward_rl_zoo3_model, zip(x0s, repeat(kwargs)))

r_ars = [a[0] for a in allres_ars]
db["td3_reward_x0_range"] = r_ars

In [ ]:
# ARS results
kwargs = {'algo': 'trpo'}


pool = mp.Pool(mp.cpu_count())
allres_ars = pool.starmap(parallel.job_get_episode_reward_rl_zoo3_model, zip(x0s, repeat(kwargs)))

r_ars = [a[0] for a in allres_ars]
db["trpo_reward_x0_range"] = r_ars

In [ ]:
# ARS results
kwargs = {'algo': 'a2c'}


pool = mp.Pool(mp.cpu_count())
allres_ars = pool.starmap(parallel.job_get_episode_reward_rl_zoo3_model, zip(x0s, repeat(kwargs)))

r_ars = [a[0] for a in allres_ars]
db["a2c_reward_x0_range"] = r_ars

#### Plots

In [ ]:
fontsize=19
fontsize_title=24

In [ ]:
cmap = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
    "#bcbd22", "#17becf", "#393b79", "#637939"
]


fig, ax1 = plt.subplots()
fig.set_figheight(8)
fig.set_figwidth(12)

ax1.plot(x0s, db["r_two"], '.', label=r'$\pi_{\text{opt}, x_0}$', color=cmap[0])
ax1.plot(x0s, db["r_pi_ana"], label=r'$\pi_{\text{ana}}$', color=cmap[1])
ax1.plot(x0s, db["ch_ars_reward_x0_range"], label=r'CH-3-ARS', color=cmap[2])
ax1.plot(x0s, db["chebyshev_reward_x0_range"], label=r'CH-3-REI', color=cmap[3])
ax1.plot(x0s, db["ch_ppo_reward_x0_range"], label=r'CH-3-PPO', color=cmap[4])
ax1.plot(x0s, db["ars_reward_x0_range"], label=r'ARS', color=cmap[5])
ax1.plot(x0s, db["sac_reward_x0_range"], label=r'SAC', color=cmap[6])
ax1.plot(x0s, db["ppo_reward_x0_range"], label=r'PPO', color=cmap[7])
ax1.plot(x0s, db["ddpg_reward_x0_range"], label=r'DDPG', color=cmap[8])
ax1.plot(x0s, db["td3_reward_x0_range"], label=r'TD3', color=cmap[9])
ax1.plot(x0s, db["trpo_reward_x0_range"], label=r'TRPO', color=cmap[10])
ax1.plot(x0s, db["a2c_reward_x0_range"], label=r'A2C', color=cmap[11])
ax1.set_xlabel(r'$x_0$', fontsize=fontsize+5)
ax1.set_ylabel(r'$R$', fontsize=fontsize+5)
#ax1.set_title(r'$R$ over $x_0$', fontsize=fontsize_title)
plt.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=4,
    fontsize=fontsize
)
ax1.tick_params(axis='both', labelsize=fontsize)
ax1.set_yticks([90, 95, 100])
ax1.set_xticks([-0.6, -0.5, -0.4])
plt.ylim(87, 100)

#fig.legend(loc='lower center', bbox_to_anchor=(0.17, 0.68))
fig.tight_layout()
#fig.suptitle(r'$R$ over different $x_0$' , y=1.1)  # otherwise the right y-label is slightly clipped
fig.savefig("mc-r-over-x0.pdf", bbox_inches='tight')

In [ ]:
x0s = np.linspace(-0.6, -0.4, 100)

env = gym.make("MountainCarContinuous-v0")
mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3,
                                                               normalize_observations=True,
                                                               mu_coeffs=db['chebyshev_reinforce_best_single_episode_result_coeffs'])
ars_model, ars_env = get_rl_zoo3_model_and_generate_env('ars')
ppo_model, ppo_env = get_rl_zoo3_model_and_generate_env('ppo')
sac_model, sac_env = get_rl_zoo3_model_and_generate_env('sac')
chars_model, chars_env = exp_run.get_sb3_polynomial_model_and_eval_env(algo='ars', coeffs=db['mountaincar_ch_ars_best_agent_params'])
chppo_model, chppo_env = exp_run.get_sb3_polynomial_model_and_eval_env(algo='ppo', coeffs=db['mountaincar_ch_ppo_best_agent_params'])

chebyshev_observations = db['chrei_observations_055']
chebyshev_reward = db['chrei_reward_055']

pi_ana_observations = db['pi_ana_observations_055']
pi_ana_reward = db['pi_ana_reward_055']

ars_observations = db['ars_observations_055']
ars_reward = db['ars_reward_055']

ppo_observations = db['ppo_observations_055']
ppo_reward = db['ppo_reward_055']

sac_observations = db['sac_observations_055']
sac_reward = db['sac_reward_055']

chars_observations = db['chars_observations_055']
chars_reward = db['chars_reward_055']

chppo_observations = db['chppo_observations_055']
chppo_reward = db['chppo_reward_055']

r_two = db['r_two']
r_pi_ana = db['r_pi_ana']

In [ ]:
fontsize=19
fontsize_title=24

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4)
fig.set_figwidth(30)
fig.set_figheight(6)

plot_analytic_solution_heatmap(ax=ax2, fig=fig, title=r'$\pi_{\text{ana}}$', c1=db['c1_pi_ana'], c2=db["c2_optimum"], trajectory=pi_ana_observations, trajectory_label=f'R = {pi_ana_reward:.4f}', actionbar=False, fontsize=fontsize, fontsize_title=fontsize_title, xticks=[-1.0, -0.35, 0.3], yticks=[-0.06, 0.0, 0.06], yticksright=True)
plot.plot_sb3_mountaincar_policy(chppo_model, ax=ax3, title=r'CH-3-PPO', normalized_env=True, denormalize=True, trajectory=chppo_observations, trajectory_label=f'R = {chppo_reward:.4f}', actionbar=False, fontsize=fontsize, fontsize_title=fontsize_title, xticks=[-1.0, -0.35, 0.3], yticks=[-0.06, 0.0, 0.06], hideyticks=True)
cbar = plot_rl_zoo3_policy(ars_env, ars_model, ax=ax4, fig=fig, title=r'ARS', unnormalized_ticks=True, trajectory=ars_observations, trajectory_label=f'R = {ars_reward:.4f}', actionbar=True, fontsize=fontsize, fontsize_title=fontsize_title, xticks=[-1.0, -0.35, 0.3], yticks=[-0.06, 0.0, 0.06], hideyticks=True)

ax1.plot(x0s, r_two, '.', label=r'$\pi_{\text{opt}, x_0}$')
ax1.plot(x0s, r_pi_ana, label=r'$\pi_{\text{ana}}$')
ax1.plot(x0s, db["ch_ars_reward_x0_range"], label=r'CH-3-ARS')
ax1.plot(x0s, db["ch_ppo_reward_x0_range"], label=r'CH-3-PPO')
ax1.plot(x0s, db["ars_reward_x0_range"], label=r'ARS')
ax1.set_xlabel(r'$x_0$', fontsize=fontsize+5)
#ax1.set_ylabel(r'$R$', fontsize=fontsize)
ax1.set_title(r'$R$ over $x_0$', fontsize=fontsize_title)
ax1.legend(loc='best', fontsize=fontsize)
ax1.tick_params(axis='both', labelsize=fontsize)
ax1.set_yticks([93, 96, 100])
ax1.set_xticks([-0.6, -0.5, -0.4])
ax1.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax2.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax3.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax4.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

fig.tight_layout(pad=0.5, w_pad=0.3)
pos = ax4.get_position()
ax4.set_position([pos.x0 - 0.02, pos.y0, pos.width, pos.height])
cbar_ax = cbar.ax
cbar_pos = cbar_ax.get_position()
cbar_ax.set_position([cbar_pos.x0 - 0.02, cbar_pos.y0, cbar_pos.width, cbar_pos.height])
fig.savefig("policies.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots()
fig.set_figwidth(10)
fig.set_figheight(8)
plot.plot_sb3_mountaincar_policy(chppo_model, ax=ax, title=r'Ch-3-PPO', normalized_env=True, denormalize=True, trajectory=chppo_observations, trajectory_label=f'R = {chppo_reward:.4f}', actionbar=True, fig=fig, fontsize=fontsize, fontsize_title=fontsize_title, xticks=[-1.0, -0.35, 0.3], yticks=[-0.06, 0.0, 0.06], hideyticks=False, yticksright=True)

In [ ]:
fig = plt.figure(figsize=(30, 18))
gs = gridspec.GridSpec(
    2, 4,
    width_ratios=[1, 1, 1, 0.05],  # last column = colorbars
    wspace=0.25
)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])
cax3 = fig.add_subplot(gs[0, 3])

ax4 = fig.add_subplot(gs[1, 0])
ax5 = fig.add_subplot(gs[1, 1])
ax6 = fig.add_subplot(gs[1, 2])
cax6 = fig.add_subplot(gs[1, 3])

plot.plot_sb3_mountaincar_policy(chars_model, ax=ax1, title=r'CH-3-ARS', normalized_env=True, denormalize=True, trajectory=chars_observations, trajectory_label=f'R = {chars_reward:.4f}', actionbar=False, fontsize=fontsize, fontsize_title=fontsize_title, xticks=[-1.0, -0.35, 0.3], yticks=[-0.06, 0.0, 0.06], hideyticks=False, yticksright=True, hideylabel=False)
polynomial_agents.plot_heatmap(mrp, fig=fig, ax=ax2, title=r'CH-3-REI', unnormalize=True, trajectory=chebyshev_observations, trajectory_label=f'R = {chebyshev_reward:.4f}', actionbar=False, fontsize=fontsize, fontsize_title=fontsize_title, xticks=[-1.0, -0.35, 0.3], yticks=[-0.06, 0.0, 0.06], hideyticks=True)
plot.plot_sb3_mountaincar_policy(chppo_model, fig=fig, ax=ax3, title=r'CH-3-PPO', normalized_env=True, denormalize=True, trajectory=chppo_observations, trajectory_label=f'R = {chppo_reward:.4f}', actionbar=True, actionbaraxis=cax3, fontsize=fontsize, fontsize_title=fontsize_title, xticks=[-1.0, -0.35, 0.3], yticks=[-0.06, 0.0, 0.06], hideyticks=False)
plot_rl_zoo3_policy(ars_env, ars_model, ax=ax4, fig=fig, title=r'ARS', unnormalized_ticks=True, trajectory=ars_observations, trajectory_label=f'R = {ars_reward:.4f}', actionbar=False, fontsize=fontsize, fontsize_title=fontsize_title, xticks=[-1.0, -0.35, 0.3], yticks=[-0.06, 0.0, 0.06], hideyticks=False, yticksright=True, hideylabel=False)
plot_rl_zoo3_policy(sac_env, sac_model, ax=ax5, fig=fig, title=r'SAC', unnormalized_ticks=True, trajectory=sac_observations, trajectory_label=f'R = {sac_reward:.4f}', actionbar=False, fontsize=fontsize, fontsize_title=fontsize_title, xticks=[-1.0, -0.35, 0.3], yticks=[-0.06, 0.0, 0.06], hideyticks=True)
plot_rl_zoo3_policy(ppo_env, ppo_model, ax=ax6, fig=fig, title=r'PPO', unnormalized_ticks=True, trajectory=ppo_observations, trajectory_label=f'R = {ppo_reward:.4f}', actionbar=True, actionbaraxis=cax6, fontsize=fontsize, fontsize_title=fontsize_title, xticks=[-1.0, -0.35, 0.3], yticks=[-0.06, 0.0, 0.06], hideyticks=False)

ax1.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax2.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax3.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax4.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax5.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax6.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

#fig.tight_layout(pad=0.5, w_pad=0.3)
# pos = ax3.get_position()
# ax3.set_position([pos.x0 - 0.02, pos.y0, pos.width, pos.height])
# cbar_ax1 = cbar1.ax
# cbar_pos1 = cbar_ax1.get_position()
# cbar_ax1.set_position([cbar_pos1.x0 - 0.02, cbar_pos1.y0, cbar_pos1.width, cbar_pos1.height])
# pos = ax6.get_position()
# ax6.set_position([pos.x0 - 0.02, pos.y0, pos.width, pos.height])
# cbar_ax2 = cbar2.ax
# cbar_pos2 = cbar_ax2.get_position()
# cbar_ax2.set_position([cbar_pos2.x0 - 0.02, cbar_pos2.y0, cbar_pos2.width, cbar_pos2.height])
pos = cax3.get_position()
cax3.set_position([
    pos.x0 - 0.01,  # move cbar left
    pos.y0,
    pos.width,
    pos.height
])
pos = cax6.get_position()
cax6.set_position([
    pos.x0 - 0.01,  # move cbar left
    pos.y0,
    pos.width,
    pos.height
])
fig.savefig("morepolicies.pdf", bbox_inches='tight')

In [ ]:
c1 = find_c1(-0.55, db["c2_optimum"])
res = run_analytic_policy(c1=c1, c2=db["c2_optimum"], start=start)
observations = res[-1]

In [ ]:
c1

In [ ]:
fig, ax1 = plt.subplots() 
fig.set_figwidth(10)
fig.set_figheight(8)
#fig.tight_layout()
plot_analytic_solution_heatmap(ax=ax1, fig=fig, c1=c1, c2=db["c2_optimum"], trajectory=observations, trajectory_label=f'R = {res[0]:.4f}')
fig.savefig(".\\paper-figures\\policyheatmap_analytic_trajectory.pdf")

In [ ]:
start_loc = -0.55
kwargs = {'start_loc': start_loc, 'single_phase': True}
v_max = 0.07
maxc = 1.0 / v_max

c = np.linspace(0.01, maxc, 1001)

cins = [[c, 0.0] for c in c]

pool = mp.Pool(mp.cpu_count())
allres_c = pool.starmap(parallel.job_analytic_solution_policy, zip(cins, repeat(kwargs)))

In [ ]:
db["single_phase_c_allres"] = allres_c

In [ ]:
allres_c = db["single_phase_c_allres"]
cs = [a[0][0] for a in allres_c]
ls = [(100-a[1])/0.1 for a in allres_c]
target_velocities = [100*a[3] for a in allres_c]
obs = [a[4] for a in allres_c]

nan_indices = [i for i, x in enumerate(ls) if np.isnan(x)]
if nan_indices:
    start_index = nan_indices[-1] + 1
else:
    start_index = 0

pos = [[[t[0] for t in a], target_velocities[i], cs[i]] for i, a in enumerate(obs) if isinstance(a, list)]
minpos = [[min(a[0]), a[1], a[2]] for a in pos]

fontsize=20
fontsize_title=24

fig, ax1 = plt.subplots()
fig.set_figwidth(18)

ax1.plot([row[2] for row in minpos], [row[0] for row in minpos], '.r', label=r'$x_{k-1}$')
ax1.set_xlabel(r'$C$', fontsize=fontsize)
ax1.set_ylabel(r'$x_{k-1}$', fontsize=fontsize)
ax1.tick_params(axis='both', labelsize=fontsize)
plt.axhline(y=-1.2, c='gray', linestyle='dotted', label=r'$x_{\text{min}}$', alpha=0.5)
plt.axhline(y=start_loc, c='gray', linestyle='dashdot', label=r'$x_{0}$', alpha=0.5)

ax2 = ax1.twinx()
ax2.plot([row[2] for row in minpos], [row[1] for row in minpos], '.', label=r'$100 \cdot \dot{x}(t_*)$')
ax2.plot(c, ls, '.', label=r'$\ell$')
ax2.set_xlabel(r'$C$')
ax2.tick_params(axis='both', labelsize=fontsize)
#ax2.set_ylabel(r'$\dot{x}(t_*)$')
ax2.set_ylabel(r'$\ell, 100 \cdot \dot{x}(t_*)$', fontsize=fontsize)

fig.legend(loc='upper left', bbox_to_anchor=(0.07, 0.95), fontsize=fontsize)
fig.tight_layout()
#fig.suptitle(r'$x_{k-1}$ and excessive velocity $\dot{x}(t_*)$ over $C$ at $x_0=$' f' {start_loc}.' '\n' r'Left wall at $x~=-1.5$', y=1.1)  # otherwise the right y-label is slightly clipped
plt.savefig("single_phase_c.pdf", bbox_inches='tight')